# MV-SAM3D — Hallucination Evaluation of the Base Model

We evaluate the **base MV-SAM3D** generator (both stages) with the three label-free
hallucination predictors from *Hallucination in World Models is Predictable and
Preventable* (Hansen & Wang, 2026 — https://www.nicklashansen.com/mmbench2/).

Those predictors were derived for a **video world model**, but each one maps cleanly onto
MV-SAM3D's **flow-matching 3D generator**. This notebook adapts and computes all three on a
single scene, using our usual Drive input (separate `ground/` and `satellite/` folders) but
**merging every view into one multi-view set** so the model sees the whole dataset at once.


## 0 — How the three predictors map onto MV-SAM3D

MV-SAM3D generates 3D in two flow-matching stages (rectified flow, Euler ODE solver):

| | latent it denoises | decoder | encoder | output |
|---|---|---|---|---|
| **Stage 1 (SS)** | dense `z_SS` = `(B, 4096, 8)` ≡ `16³×8` | `ss_decoder` → `64³` occupancy | `ss_encoder` (VAE `E_S`) | coarse voxel structure |
| **Stage 2 (SLAT)** | sparse `z_SLAT` = `(B, M, 8)` on the SS coords | `slat_decoder_*` → mesh/gaussians | (image-feature encoder; no clean latent inverse) | textured mesh |

The paper's three metrics and our adaptation:

- **`u_r` — tokenizer round-trip residual.** Paper: `u_r = ‖ẑ − Enc(Dec(ẑ))‖`. On-manifold
  latents survive a decode→encode round trip; hallucinated ones drift. **Here:** we take the
  generated **SS** latent `ẑ`, decode it to `64³` occupancy, re-encode with the SS VAE encoder
  `E_S`, and measure the latent residual. (The generator emits latents in the *same* space the
  decoder consumes — no de-normalization in `sample_sparse_structure` — so the round trip is
  well-posed. SLAT is skipped for `u_r`: its decoder emits meshes/gaussians, not something the
  SLAT encoder re-ingests, and SLAT latents are additionally rescaled by `slat_std/slat_mean`.)

- **`u_s` — inter-seed variance.** Paper: how much the prediction varies across independent
  denoising seeds. **Here:** we re-run generation with different **initial Gaussian latents**
  (`torch.manual_seed`) and measure per-voxel dispersion — of the `64³` occupancy for Stage 1,
  and of the SLAT features on **fixed** SS coords for Stage 2. High-variance voxels are exactly
  the geometry the model is unsure about.

- **`u_f` — flow instability.** Paper: how much the denoiser's *clean-frame prediction* moves
  between Euler substeps; a well-conditioned step settles fast, an under-conditioned one keeps
  oscillating. **Here:** rectified flow gives a closed-form clean-data prediction
  `x̂₁ = x_t + (1−t)·v(x_t,t)`. We log every Euler step of the sampler (by wrapping the ODE
  solver) and track `‖Δx̂₁‖` across steps (dynamism-normalized). We compute it for **both** stages.

We report each metric **after Stage 1** and **after both stages**, as requested. Because the
sampler is deterministic given a fixed seed, all comparisons are reproducible.


## 1 — Environment & install (self-contained, fresh Colab A100-80GB)

This section is **fully self-contained** — run it top to bottom in a brand-new
Colab session. It is the exact, known-working install sequence from
`mvsam3d_example_current.ipynb`.

**Before you start:**
1. **Runtime → Change runtime type → A100 GPU** (80 GB). The SLAT backbone + DA3
   on building imagery need the headroom.
2. **Add your Hugging Face token to Colab Secrets** as `HF_TOKEN` (sidebar key
   icon, toggle notebook access). `facebook/sam-3d-objects` is a **gated** repo —
   request access on its HF page first, or checkpoint download will 401.

**Restart gotcha (important):** Cell *1.2 (Step 0)* pins `torch==2.5.1+cu121`. On
a fresh runtime Colab ships a newer torch, so this cell will downgrade it and
print *"Restart the runtime now!"*. When you see that: **Runtime → Restart
session**, then re-run **from the clone cell (1.3) downward** (skip re-running
Step 0 — torch is already correct, and it must not be re-pinned).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch, subprocess, sys, os

assert torch.cuda.is_available(), "No GPU — enable a GPU runtime first!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:     {gpu_name}")
print(f"VRAM:    {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"Python:  {sys.version}")

if vram_gb < 30:
    print("\n⚠  Less than 32 GB VRAM — inference may OOM on large batches. "
          "Consider switching to an A100 runtime.")

# SAM 3D Objects targets torch 2.5.1 + CUDA 12.1.
# Colab's default torch is often too new for pre-built wheels of
# flash-attn, kaolin, spconv, etc.  Pin the correct stack now.
TARGET_TORCH = "2.5.1"
if not torch.__version__.startswith(TARGET_TORCH):
    print(f"\n▶ Downgrading torch {torch.__version__} → {TARGET_TORCH}+cu121 …")
    !pip install -q \
        torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
        --index-url https://download.pytorch.org/whl/cu121
    !pip install -q xformers==0.0.28.post3
    print("✓ Torch stack pinned.  ⚠ Restart the runtime now!")
    print("   Go to Runtime → Restart session, then re-run from this cell.")
else:
    print(f"\n✓ torch {torch.__version__} is the target version")

> **If Step 0 above printed "Restart the runtime now!":** do
> **Runtime → Restart session**, then continue from this cell (1.3). Do **not**
> re-run Step 0 after the restart. If it printed *"torch 2.5.1 is the target
> version"*, just keep going — no restart needed.

In [ ]:
# 1.1 — Define paths and clone the three repos (same remotes as the working notebook)
from pathlib import Path
import os, sys

ROOT         = Path("/content")
SAM3D_ROOT   = ROOT / "sam-3d-objects"
DA3_ROOT     = ROOT / "Depth-Anything-3"
MVSAM3D_ROOT = ROOT / "MV-SAM3D"

# Pin numpy AND scipy for EVERY pip install in this session. SAM3D + DA3 require
# numpy<2, but kaolin/scipy/etc. try to pull numpy 2.x — and upgrading numpy while it
# is already imported corrupts the live kernel ("No module named 'numpy.char' / fails
# sanity checks"). PIP_CONSTRAINT makes pip refuse to change them.
# scipy is pinned too: modern Colab ships a numpy-2-only scipy whose array-API shim
# does `from numpy import strings` (numpy>=2 only), so under numpy 1.26 `import scipy`
# explodes with "No module named 'numpy.strings'" — which crashes `import kaolin`.
# scipy 1.13.1 supports numpy 1.26 and has no such code path.
# NOTE: env vars do NOT survive a runtime restart — this cell re-sets it, so always
# re-run it (it's the first cell after Step 0) before the install cells.
_np_pin = ROOT / "_numpy_pin.txt"
_np_pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
os.environ["PIP_CONSTRAINT"] = str(_np_pin)
print("PIP_CONSTRAINT ->", os.environ["PIP_CONSTRAINT"], "(numpy 1.26.4 + scipy 1.13.1 locked)")

# Force-align scipy with numpy 1.26.4 NOW, before anything imports scipy this session.
# (PIP_CONSTRAINT prevents future upgrades but won't downgrade the scipy Colab preinstalled.)
# Doing it here — on a fresh kernel, before any `import scipy` — avoids stale broken
# modules; otherwise kaolin's import later dies on "No module named 'numpy.strings'".
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
import importlib, scipy, numpy
importlib.reload(scipy)
print(f"✓ numpy {numpy.__version__}  scipy {scipy.__version__}")

%cd /content
REPOS = {
    "sam-3d-objects":   "https://github.com/facebookresearch/sam-3d-objects.git",
    "Depth-Anything-3": "https://github.com/ByteDance-Seed/Depth-Anything-3.git",
    "MV-SAM3D":         "https://github.com/devinli123/MV-SAM3D.git",
}
for name, url in REPOS.items():
    if os.path.isdir(name):
        print(f"OK   {name}/ already exists")
    else:
        !git clone --depth 1 {url}
        print(f"cloned {name}")


In [ ]:
# xformers must match the installed torch+cuda version.
# Colab usually ships a compatible pair, but let's be explicit.
!pip install -q xformers

# Install DA3 as an editable package (pulls in its own deps)
!pip install -q -e /content/Depth-Anything-3

print("\n✓ Depth Anything 3 installed")

In [ ]:
import torch

TORCH_VERSION = torch.__version__.split("+")[0]   # e.g. "2.10.0"
CUDA_TAG      = torch.version.cuda.replace(".", "")  # e.g. "128"
print(f"Targeting torch {TORCH_VERSION}  CUDA {torch.version.cuda}")

# ── 4a: Install the sam3d_objects package (code only, no deps) ──
!pip install -q --no-deps -e /content/sam-3d-objects

# ── 4b: Runtime deps (open3d and utils3d conflict on numpy, install separately) ──
!pip install -q \
    hydra-core==1.3.2 \
    omegaconf \
    loguru \
    opencv-python-headless \
    trimesh \
    roma \
    seaborn \
    safetensors \
    tqdm \
    einops \
    astor \
    lightning \
    xatlas \
    pyvista \
    pymeshfix \
    igraph \
    "numpy<2" \
    "huggingface_hub[cli]" \
    gradio

# open3d and utils3d have conflicting numpy pins — install each alone
!pip install -q open3d
!pip install -q --no-deps "utils3d @ git+https://github.com/EasternJournalist/utils3d.git@3913c65d81e05e47b9f367250cf8c0f7462a0900"

# ── 4c: MoGe (pointmap geometry utilities used by sam3d_objects) ──
!pip install -q --no-deps "MoGe @ git+https://github.com/microsoft/MoGe.git@a8c37341bc0325ca99b9d57981cc3bb2bd3e255b"

# The long list of "sam3d-objects requires X" warnings above is expected —
# we intentionally skipped those with --no-deps. They are dev/training deps
# not needed for inference.

print("\n✓ Core SAM 3D deps installed")

In [ ]:
import torch, subprocess, sys, os

TORCH_VER = torch.__version__.split("+")[0]            # "2.5.1"
CUDA_VER  = "cu" + torch.version.cuda.replace(".", "") # "cu121"
PY_VER    = f"{sys.version_info.major}{sys.version_info.minor}"

print(f"torch {TORCH_VER}  CUDA {torch.version.cuda}  Python {PY_VER}")
assert TORCH_VER.startswith("2.5"), (
    f"Expected torch 2.5.x but got {TORCH_VER}. "
    "Re-run Step 0 to pin the correct version, then restart the runtime."
)

# ── Re-assert the numpy/scipy pin for THIS cell (kaolin is the usual offender) ──
# Every `!pip install` below must honor numpy<2. If this cell runs on a kernel where
# PIP_CONSTRAINT isn't set (e.g. cell 1.1 wasn't re-run after a restart), `pip install
# kaolin` pulls numpy 2.x as a dependency and overwrites the numpy already loaded in
# this kernel -> 'No module named numpy.char' / 'fails sanity checks'. Recreate it here.
from pathlib import Path as _Path
_pin = _Path(os.environ.get("PIP_CONSTRAINT") or "")
if not (_pin and _pin.exists()):
    _pin = _Path("/content/_numpy_pin.txt")
    _pin.write_text("numpy==1.26.4\nscipy==1.13.1\n")
os.environ["PIP_CONSTRAINT"] = str(_pin)
print("PIP_CONSTRAINT ->", os.environ["PIP_CONSTRAINT"], "(numpy 1.26.4 + scipy 1.13.1 locked)")

!pip install -q ninja packaging

# ── Safety check: scipy must be numpy-1.26 compatible before kaolin imports it ──
# (cell 1.1 already downgrades scipy to 1.13.1; this just verifies / repairs.)
import scipy as _scipy
if not _scipy.__version__.startswith(("1.10", "1.11", "1.12", "1.13")):
    print(f"scipy {_scipy.__version__} is too new for numpy 1.26 — fixing …")
    !pip install -q "scipy==1.13.1"
    print("scipy downgraded. If `import kaolin` below still fails, RESTART the runtime")
    print("and re-run from cell 1.1 (scipy was already imported in this kernel).")
else:
    print(f"✓ scipy {_scipy.__version__} (numpy-1.26 compatible)")

# ── PyTorch3D ──
try:
    import pytorch3d
    print(f"✓ pytorch3d {pytorch3d.__version__}")
except ImportError:
    print("Installing PyTorch3D …")
    ret = subprocess.run(
        f"pip install --no-index --no-cache-dir pytorch3d "
        f"-f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/"
        f"py{PY_VER}_{CUDA_VER}_pyt{TORCH_VER.replace('.','')}/download.html",
        shell=True,
    )
    if ret.returncode != 0:
        print("Pre-built wheel not found — building from source (5-10 min) …")
        !pip install "pytorch3d @ git+https://github.com/facebookresearch/pytorch3d.git"
    import pytorch3d
    print(f"✓ pytorch3d {pytorch3d.__version__}")

# ── flash-attn (pre-built wheel exists for torch 2.5.1 + cu121) ──
try:
    import flash_attn
    print(f"✓ flash_attn {flash_attn.__version__}")
except ImportError:
    print("Installing flash-attn …")
    !pip install -q flash-attn --no-build-isolation
    import flash_attn
    print(f"✓ flash_attn {flash_attn.__version__}")

# ── kaolin (NVIDIA publishes wheels indexed by torch+cuda) ──
os.environ["PIP_FIND_LINKS"] = (
    f"https://nvidia-kaolin.s3.us-east-2.amazonaws.com/"
    f"torch-{TORCH_VER}_{CUDA_VER}.html"
)
try:
    import kaolin
    print(f"✓ kaolin {kaolin.__version__}")
except ImportError:
    print("Installing kaolin …")
    # --constraint is passed explicitly (belt-and-suspenders with PIP_CONSTRAINT) so the
    # kaolin dependency resolver can NEVER pull numpy 2.x into this live kernel.
    !pip install -q kaolin --constraint "$PIP_CONSTRAINT"
    # REPAIR numpy after kaolin's deps. Even with the version pinned to 1.26.4, kaolin's
    # dependency install can leave a MIXED numpy on disk: 1.26 metadata (so a version
    # check passes) but a numpy.random C-extension rebuilt against numpy 2.x. Importing
    # it then dies with 'numpy.dtype size changed ... Expected 96 ... got 88'. A clean
    # --force-reinstall rewrites EVERY numpy binary to a consistent 1.26.4 before import.
    !pip install -q --force-reinstall --no-deps "numpy==1.26.4"
    try:
        import kaolin
        print(f"✓ kaolin {kaolin.__version__}")
    except (ImportError, ValueError) as _e:
        raise RuntimeError(
            f"kaolin import still failing after numpy repair ({type(_e).__name__}: {_e}).\n"
            "This kernel already loaded a broken numpy, so it cannot recover in place.\n"
            "FIX: Runtime > Restart session, then run cell 1.1 FIRST, then re-run this cell.\n"
            "(The disk is now repaired, so the next run will import kaolin cleanly.)") from _e

# ── gsplat (Gaussian splatting renderer) ──
try:
    import gsplat
    print(f"✓ gsplat {getattr(gsplat, '__version__', 'ok')}")
except ImportError:
    print("Installing gsplat …")
    !pip install -q --no-build-isolation \
        "gsplat @ git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70"
    import gsplat
    print(f"✓ gsplat installed")

# ── spconv (sparse convolution, used by the 3D backbone) ──
# Two failure modes we have to defeat here, in order:
#  (1) ABI mismatch: spconv-cu121==2.3.8 is compiled against cumm 0.7.x and pins
#      cumm-cu121<0.8.0. Letting pip pull the *latest* cumm (>=0.8.0) gives a
#      core_cc that fails with 'could not convert default argument workspace:
#      tv::Tensor ... type not registered yet'. Fix: pin BOTH in one resolve.
#  (2) Bogus JIT recompile: on Colab, cumm misdetects an 'editable' install and
#      tries to recompile core_cc at import. That compile fails because cumm
#      probes /usr/local/lib/python3.12/dist-packages/include (exists but empty)
#      as its header root instead of cumm/include, so it can't find
#      tensorview/*.h. Fix: CUMM_DISABLE_JIT=1 / SPCONV_DISABLE_JIT=1 force use
#      of the prebuilt core_cc shipped in the wheel — no compile at all.
# These env vars are read once at first import of cumm.constants, so they MUST be
# set before anything imports cumm/spconv (true on a fresh kernel).
os.environ["CUMM_DISABLE_JIT"]   = "1"
os.environ["SPCONV_DISABLE_JIT"] = "1"
# Probe spconv.pytorch (NOT bare `import spconv`) — bare import is lazy and never
# loads the C extension, so it can't detect a broken binary.
try:
    import spconv.pytorch  # noqa: F401 — this is what actually loads core_cc
    print("✓ spconv installed")
except Exception:
    print("Installing spconv (pinned pair: spconv-cu121==2.3.8 + cumm-cu121==0.7.11) …")
    !pip install -q --force-reinstall "spconv-cu121==2.3.8" "cumm-cu121==0.7.11"
    # Drop any half-loaded spconv/cumm modules so the freshly installed binary is
    # picked up; if core_cc was never loaded this lets us recover without a restart.
    import sys as _sys
    for _m in [k for k in list(_sys.modules) if k == "spconv" or k.startswith("spconv.") or k == "cumm" or k.startswith("cumm.")]:
        del _sys.modules[_m]
    try:
        import spconv.pytorch  # noqa: F401
        print("✓ spconv installed (matched pair, JIT disabled)")
    except Exception as _e:
        raise RuntimeError(
            f"spconv still failing after pinning spconv-cu121==2.3.8 + cumm-cu121==0.7.11 "
            f"({type(_e).__name__}: {_e}).\n"
            "This kernel already imported a broken/JIT cumm, so it can't recover in place.\n"
            "FIX: Runtime > Restart session, then run cell 1.1 FIRST, then re-run this cell.\n"
            "(The disk now has the correct matched pair + JIT disabled, so the next run imports cleanly.)") from _e

# Final numpy repair: spconv/cumm force-reinstalls above can also re-link numpy, so do a
# clean rewrite (not a no-op version pin) to guarantee consistent 1.26.4 binaries on disk.
!pip install -q --force-reinstall --no-deps "numpy==1.26.4"
print("\n✓ Compiled extensions done (numpy force-reinstalled to a clean 1.26.4)")

In [ ]:
import os

# Colab has no CONDA_PREFIX — point CUDA_HOME to the system CUDA toolkit.
# inference.py does: os.environ["CUDA_HOME"] = os.environ["CONDA_PREFIX"]
# We pre-set both so that line doesn't crash.
cuda_home = "/usr/local/cuda"
os.environ["CUDA_HOME"]    = cuda_home
os.environ["CONDA_PREFIX"] = cuda_home
os.environ["LIDRA_SKIP_INIT"] = "true"

# Run the Hydra patch shipped with sam-3d-objects
!python /content/sam-3d-objects/patching/hydra
print("✓ Hydra patched")

### 1.7 — Hugging Face auth + checkpoints

Logs in with the `HF_TOKEN` Colab secret, then downloads the **SAM 3D Objects**
checkpoint (gated) and the **Depth-Anything-3** model. The SAM 3D download lands
in `sam-3d-objects/checkpoints/hf/` and is symlinked into
`MV-SAM3D/checkpoints/hf` in Section 9 so `pipeline.yaml` resolves.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Retrieve the token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Login programmatically, bypassing the broken widget
login(token=hf_token)

In [ ]:
from huggingface_hub import snapshot_download
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

dl_path = snapshot_download(
    repo_id="facebook/sam-3d-objects",
    token=hf_token,
    local_dir=str(SAM3D_ROOT / "checkpoints" / "hf"),
)
print(f"✓ SAM 3D checkpoint saved to {dl_path}")
!ls -lh {SAM3D_ROOT / "checkpoints" / "hf"}

In [ ]:
from huggingface_hub import snapshot_download

DA3_MODEL_ID = "depth-anything/DA3NESTED-GIANT-LARGE"
da3_cache = snapshot_download(repo_id=DA3_MODEL_ID)
print(f"✓ DA3 model cached at {da3_cache}")

In [ ]:
import sys, os

os.environ.setdefault("CUDA_HOME",       "/usr/local/cuda")
os.environ.setdefault("CONDA_PREFIX",    "/usr/local/cuda")
os.environ["LIDRA_SKIP_INIT"] = "true"

# MV-SAM3D root (so `from sam3d_objects.…` resolves to MV-SAM3D's copy)
sys.path.insert(0, str(MVSAM3D_ROOT))
# MV-SAM3D notebook helpers (inference.py, load_images_and_masks.py, mesh_alignment.py)
sys.path.insert(0, str(MVSAM3D_ROOT / "notebook"))
# DA3 source directory
sys.path.insert(0, str(DA3_ROOT / "src"))

# Smoke tests
from depth_anything_3.api import DepthAnything3;    print("✓ depth_anything_3")
from inference import Inference;                     print("✓ SAM 3D Inference")
from load_images_and_masks import load_images_and_masks_from_path; print("✓ load_images_and_masks")
from sam3d_objects.utils.cross_attention_logger import CrossAttentionLogger; print("✓ CrossAttentionLogger")
from sam3d_objects.utils.latent_weighting import WeightingConfig; print("✓ LatentWeightManager")
from pytorch3d.transforms import Transform3d;       print("✓ pytorch3d")

print("\n✓ All imports OK")

In [ ]:
# 1.9 — Extra deps (cv2 already ships with the SAM 3D stack). No OSM/shapely here:
# this notebook has no footprint step, so those heavy geo deps are not installed.
!pip install -q numpy==1.26.4   # re-pin numpy AFTER any installs above

import numpy as np, cv2
print("numpy", np.__version__, "| cv2", cv2.__version__)


## 2 — Configuration & hallucination knobs

In [ ]:
# 2.0 — Paths and hyper-parameters  (EDIT the <-- marked lines)
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/SatelliteFuse/datasets/va_split")   # <-- EDIT
GROUND_DIR = DATA_ROOT / "ground"        # ground-level views
AERIAL_DIR = DATA_ROOT / "satellite"     # aerial / near-nadir views
MASK_NAME   = "masks"                    # <-- mask subfolder name (e.g. ground/masks/)
TEXT_PROMPT = "building"                 # <-- SAM3 text prompt used to find the object

OUTPUT_DIR = Path("/content/mvsam3d_halluc_out");  OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HALLUC_OUT = OUTPUT_DIR / "hallucination";         HALLUC_OUT.mkdir(parents=True, exist_ok=True)

# ---- Base MV-SAM3D inference parameters (both stages run on ALL views) ----
SEED         = 0
STAGE1_STEPS = 25      # sparse-structure denoising steps
STAGE2_STEPS = 25      # SLAT denoising steps
MODE         = "multidiffusion"
SS_WEIGHTING     = True    # base MV-SAM3D uses entropy weighting for Stage 1
SS_ENTROPY_ALPHA = 60.0
SS_WARMUP_STEPS  = 1

# ---- Hallucination-eval knobs ----
SEED_LIST = list(range(10))   # initial-latent seeds for inter-seed variance u_s (>=3; 10 = tighter estimate)
UF_STEPS  = STAGE1_STEPS   # steps used for the flow-instability (u_f) probe
RUN_SLAT_METRICS = True    # also evaluate Stage 2 (SLAT); False = faster geometry-only run

print("Ground dir:", GROUND_DIR)
print("Aerial dir:", AERIAL_DIR)
print(f"Steps: stage1={STAGE1_STEPS} stage2={STAGE2_STEPS} | base seed={SEED} | mode={MODE}")
print(f"Inter-seed set: {SEED_LIST} | SLAT metrics: {RUN_SLAT_METRICS}")


## 2.5 — Segment BOTH view sets with SAM3

Both stages need a **per-view building mask**. This section creates them for
**ground** and **aerial** folders the same way:

1. **Organize** — drop raw photos straight into `ground/` and `satellite/`. We
   copy each into `<folder>/images/`.
2. **(Optional) Interactive points** — click the ground views to disambiguate the
   instance (**left-click = keep**, **right-click = exclude**). Skip to use the
   `TEXT_PROMPT` text prompt only. (The picker targets the ground set; aerial uses
   the text prompt.)
3. **Segment** — SAM3 writes RGBA masks to `<folder>/<MASK_NAME>/<stem>.png` for
   both folders. Existing masks are kept unless `FORCE_RESEGMENT=True`.

Masks are matched to images **by stem**, so original names/numbers are preserved.

> SAM3's weights (~3.4 GB) stream from Hugging Face on first use — make sure the
> HF-login cell in Section 1 ran successfully.


In [ ]:
# 2.5.1 — Install SAM3 (segmentation) and fix its tokenizer asset path
!pip install -q --no-deps sam3
!pip install -q decord ftfy iopath
!pip install -q numpy==1.26.4          # keep numpy pinned (PIP_CONSTRAINT enforces this too)

# SAM3's text encoder loads a BPE vocab from dist-packages/assets/. Locate & link it.
import glob, os, urllib.request
_target = "/usr/local/lib/python3.12/dist-packages/assets"
os.makedirs(_target, exist_ok=True)
_dst = f"{_target}/bpe_simple_vocab_16e6.txt.gz"
if os.path.exists(_dst):
    print("BPE vocab already present.")
else:
    _hits = (glob.glob("/usr/local/lib/python3.12/dist-packages/sam3/**/bpe_simple*", recursive=True)
             or glob.glob("/usr/local/lib/python3.12/dist-packages/**/bpe_simple_vocab_16e6*", recursive=True))
    if _hits:
        os.symlink(_hits[0], _dst); print("Linked BPE vocab from", _hits[0])
    else:
        urllib.request.urlretrieve(
            "https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", _dst)
        print("Downloaded BPE vocab ->", _dst)

In [ ]:
# 2.5.2 — SAM3 segmenter (one shared, cached processor for picker + batch run)
import torch, numpy as np
from pathlib import Path
from PIL import Image
from loguru import logger

_sam3_model_cache = {}   # picker (_preview_mask) and the segmenter both read this

def _get_sam3_processor(confidence_threshold: float = 0.1):
    """Build (once) and cache a SAM3 image processor; weights stream from HF."""
    if "processor" not in _sam3_model_cache:
        from sam3.model_builder import build_sam3_image_model
        from sam3.model.sam3_image_processor import Sam3Processor
        print("Loading SAM3 (one-time; first run downloads ~3.4 GB)...")
        model = build_sam3_image_model(load_from_HF=True, checkpoint_path=None)
        _sam3_model_cache["processor"] = Sam3Processor(model, confidence_threshold=confidence_threshold)
        print("SAM3 ready.")
    return _sam3_model_cache["processor"]


class SAM3MultiObjectSegmenter:
    """Segment one object across many views; optional per-view point disambiguation."""

    def __init__(self, processor=None, confidence_threshold: float = 0.1):
        self.processor = processor or _get_sam3_processor(confidence_threshold)

    @staticmethod
    def _select_mask_by_points(masks, points, labels, image_size):
        w, h = image_size
        best_idx, best_score = 0, -float("inf")
        for mi in range(len(masks)):
            m = masks[mi]
            m_np = m.squeeze(0).cpu().numpy() if torch.is_tensor(m) else (m.squeeze(0) if m.ndim > 2 else m)
            if m_np.shape != (h, w):
                m_pil = Image.fromarray((m_np * 255).astype(np.uint8)).resize((w, h), Image.NEAREST)
                m_np = np.array(m_pil) / 255.0
            m_bool = m_np > 0.5
            score = 0
            for (px, py), lbl in zip(points, labels):
                inside = m_bool[max(0, min(int(py), h - 1)), max(0, min(int(px), w - 1))]
                score += 1 if (lbl == 1) == bool(inside) else -1
            if score > best_score:
                best_score, best_idx = score, mi
        return best_idx

    def segment_object_multiview(self, images_dir, object_name, text_prompt, output_dir,
                                 per_view_points=None, image_names=None):
        images_dir, output_dir = Path(images_dir), Path(output_dir)
        logger.info(f"[Segmenting: {object_name}]  prompt='{text_prompt}'")
        if per_view_points:
            logger.info(f"  Per-view point prompts for {len(per_view_points)} view(s)")

        def _key(p):
            try:    return (0, int(p.stem), p.stem)
            except ValueError: return (1, 0, p.stem)

        all_files = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")), key=_key)
        # Map stem -> index in the FULL list so picker point keys stay valid even when we
        # only re-segment a subset of views below.
        full_index = {p.stem: idx for idx, p in enumerate(all_files)}
        image_files = all_files
        if image_names is not None:   # restrict to requested stems (e.g. only missing masks)
            _want = set(image_names)
            image_files = [p for p in all_files if p.stem in _want]
        if not image_files:
            return {"success": False, "error": "No images found", "success_views": 0, "total_views": 0}
        logger.info(f"  Processing {len(image_files)} views...")

        mask_dir = output_dir / object_name
        mask_dir.mkdir(parents=True, exist_ok=True)
        success, failed = 0, []

        for img_path in image_files:
            i = full_index[img_path.stem]   # index into the picker's full-list keys
            try:
                image = Image.open(img_path).convert("RGB")
                state = self.processor.set_image(image)
                out = self.processor.set_text_prompt(state=state, prompt=text_prompt)
                masks, scores = out["masks"], out["scores"]
                if len(masks) == 0:
                    logger.warning(f"  View {i}: no mask (confidence too low)"); failed.append(i); continue

                if per_view_points and i in per_view_points:
                    vp = per_view_points[i]
                    best_idx = self._select_mask_by_points(masks, vp["points"], vp["labels"], image.size)
                else:
                    best_idx = scores.argmax().item()

                best_mask, best_score = masks[best_idx], scores[best_idx].item()
                m_np = best_mask.squeeze(0).cpu().numpy() if torch.is_tensor(best_mask) else best_mask.squeeze(0)
                if m_np.shape != (image.size[1], image.size[0]):
                    m_np = np.array(Image.fromarray((m_np * 255).astype(np.uint8)).resize(image.size, Image.NEAREST)) / 255.0

                image_np, mask_bool = np.array(image), m_np > 0.5
                rgba = np.zeros((image_np.shape[0], image_np.shape[1], 4), dtype=np.uint8)
                rgba[mask_bool, :3] = image_np[mask_bool]
                rgba[mask_bool, 3] = 255
                Image.fromarray(rgba, "RGBA").save(mask_dir / f"{img_path.stem}.png")

                area = float((rgba[:, :, 3] > 0).mean())
                logger.info(f"  View {i}: {img_path.stem}.png  area={area*100:.1f}%  score={best_score:.3f}")
                success += 1
            except Exception as e:
                logger.error(f"  View {i}: failed - {e}"); failed.append(i)

        logger.info(f"  Result: {success}/{len(image_files)} views segmented")
        if failed:
            logger.warning(f"  Failed views: {failed}")
        return {"success": success > 0, "object_name": object_name, "total_views": len(image_files),
                "success_views": success, "failed_views": failed, "mask_dir": mask_dir}

print("SAM3MultiObjectSegmenter ready.")

In [ ]:
# 2.5.3 — Copy raw photos into <folder>/images/ for BOTH ground and aerial
import shutil
from pathlib import Path

_exts = ("*.png", "*.jpg", "*.jpeg", "*.JPG", "*.JPEG", "*.PNG", "*.webp")

def organize_images(base_dir):
    """Copy loose photos in base_dir/ into base_dir/images/ (non-destructive)."""
    base_dir = Path(base_dir)
    images_dir = base_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    flat = []
    for e in _exts:
        flat += [p for p in base_dir.glob(e) if p.is_file()]   # top-level only
    copied = 0
    for p in sorted(flat):
        dst = images_dir / p.name
        if not dst.exists():
            shutil.copy2(p, dst)
            copied += 1
    n = len(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")))
    print(f"{base_dir.name}/images/ ready: copied {copied} new file(s); {n} image(s) total")
    return images_dir, n

IMAGES_DIR, _ng = organize_images(GROUND_DIR)          # ground (also used by the picker below)
AERIAL_IMAGES_DIR, _na = organize_images(AERIAL_DIR)   # aerial
MASK_PROMPT = TEXT_PROMPT                               # the interactive picker preview reads this

assert _ng > 0, f"No ground images found. Put ground photos directly in {GROUND_DIR}"
assert _na > 0, f"No aerial images found. Put aerial photos directly in {AERIAL_DIR}"


In [ ]:
# 2.5.4 — (OPTIONAL) Interactive per-view point picker. Skip to use the text prompt only.
## Interactive Per-View Point Picker for SAM3 Segmentation
#
# All images loaded into a JS viewer. Navigate with Prev/Next.
#   LEFT CLICK = positive (green), RIGHT CLICK = negative (red)
# Click "Save All to Python" when done to store PER_VIEW_POINTS.
# Views with no points use text-only segmentation.

import base64, json
from pathlib import Path
from IPython.display import display, HTML
from google.colab import output
from PIL import Image as PILImage

def _natural_sort_key(p):
    try:
        return (0, int(p.stem), p.stem)
    except ValueError:
        return (1, 0, p.stem)

image_files = sorted(
    list(IMAGES_DIR.glob("*.png")) + list(IMAGES_DIR.glob("*.jpg")),
    key=_natural_sort_key
)
N_IMAGES = len(image_files)

# Thumbnail images for the JS viewer (resize to max 800px wide for speed)
MAX_W = 800
img_data = []
img_dims = []
for f in image_files:
    im = PILImage.open(f)
    orig_w, orig_h = im.size
    img_dims.append((orig_w, orig_h))
    if orig_w > MAX_W:
        ratio = MAX_W / orig_w
        im = im.resize((MAX_W, int(orig_h * ratio)), PILImage.LANCZOS)
    import io
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=70)
    img_data.append(base64.b64encode(buf.getvalue()).decode())

print(f"Loaded {N_IMAGES} images into viewer")

PER_VIEW_POINTS = {}

# Build JSON arrays for JS
_names_json = json.dumps([f.name for f in image_files])
_dims_json = json.dumps(img_dims)
_imgs_json = json.dumps(img_data)

def _receive_all_points(data_json):
    global PER_VIEW_POINTS
    raw = json.loads(data_json)
    PER_VIEW_POINTS = {}
    for k, v in raw.items():
        idx = int(k)
        if v:
            PER_VIEW_POINTS[idx] = {
                "points": [[p["x"], p["y"]] for p in v],
                "labels": [p["l"] for p in v],
            }
    n = sum(1 for v in PER_VIEW_POINTS.values() if v["points"])
    print(f"Saved {n}/{N_IMAGES} views with point prompts")
    for k in sorted(PER_VIEW_POINTS):
        v = PER_VIEW_POINTS[k]
        s = ", ".join(f"{'+'if l==1 else '-'}({p[0]},{p[1]})" for p, l in zip(v["points"], v["labels"]))
        print(f"  [{k}] {image_files[k].name}: {s}")

_sam3_model_cache = {}

def _preview_mask(img_idx, points_json):
    """Run SAM3 on one image, score masks with points, return overlay as base64."""
    import io, numpy as np, torch
    from PIL import Image as PILImg

    img_idx = int(img_idx)
    pts_raw = json.loads(points_json)
    if not pts_raw:
        return ""

    pts = [[p["x"], p["y"]] for p in pts_raw]
    lbls = [p["l"] for p in pts_raw]

    proc = _get_sam3_processor()
    img = PILImg.open(image_files[img_idx]).convert("RGB")
    w, h = img.size

    state = proc.set_image(img)
    out = proc.set_text_prompt(state=state, prompt=MASK_PROMPT.split(",")[0].strip())
    masks = out["masks"]
    if len(masks) == 0:
        return ""

    best_idx, best_score = 0, -float("inf")
    for mi in range(len(masks)):
        m = masks[mi]
        m_np = m.squeeze(0).cpu().numpy() if torch.is_tensor(m) else m.squeeze(0)
        if m_np.shape != (h, w):
            m_pil = PILImg.fromarray((m_np * 255).astype(np.uint8))
            m_pil = m_pil.resize((w, h), PILImg.NEAREST)
            m_np = np.array(m_pil) / 255.0
        m_bool = m_np > 0.5
        score = 0
        for (px, py), lbl in zip(pts, lbls):
            inside = m_bool[min(int(py), h-1), min(int(px), w-1)]
            score += 1 if (lbl == 1) == inside else -1
        if score > best_score:
            best_score = score
            best_idx = mi

    best_m = masks[best_idx]
    m_np = best_m.squeeze(0).cpu().numpy() if torch.is_tensor(best_m) else best_m.squeeze(0)
    if m_np.shape != (h, w):
        m_pil = PILImg.fromarray((m_np * 255).astype(np.uint8))
        m_pil = m_pil.resize((w, h), PILImg.NEAREST)
        m_np = np.array(m_pil) / 255.0

    img_np = np.array(img)
    overlay = img_np.copy()
    mask_bool = m_np > 0.5
    overlay[mask_bool] = (overlay[mask_bool] * 0.5 + np.array([255, 80, 80]) * 0.5).astype(np.uint8)

    thumb = PILImg.fromarray(overlay)
    if w > MAX_W:
        ratio = MAX_W / w
        thumb = thumb.resize((MAX_W, int(h * ratio)), PILImg.LANCZOS)
    buf = io.BytesIO()
    thumb.save(buf, format="JPEG", quality=75)
    return base64.b64encode(buf.getvalue()).decode()

output.register_callback('_receive_all_points', _receive_all_points)
output.register_callback('_preview_mask', _preview_mask)

html = f"""
<style>
#vw{{position:relative;display:inline-block;cursor:crosshair}}
#vw img{{display:block;max-width:900px}}
.mk{{position:absolute;width:18px;height:18px;border-radius:50%;border:2px solid #fff;
     transform:translate(-50%,-50%);pointer-events:none;font-size:11px;text-align:center;
     line-height:18px;color:#fff;font-weight:bold}}
.mp{{background:rgba(0,200,0,.85)}}.mn{{background:rgba(220,0,0,.85)}}
#hd{{font-size:15px;margin-bottom:6px}}
#pl{{margin-top:6px;font-family:monospace;font-size:13px;min-height:20px}}
#ct{{margin-top:8px}}
#ct button{{padding:6px 18px;margin-right:8px;font-size:14px;cursor:pointer}}
.bg{{background:#4CAF50;color:#fff;border:none;border-radius:4px}}
.bo{{background:#2196F3;color:#fff;border:none;border-radius:4px}}
</style>
<div id="hd"></div>
<div id="vw"><img id="im" /></div>
<div id="pl"></div>
<div id="ct">
  <button onclick="prev()">&larr; Prev</button>
  <button onclick="undo()">Undo</button>
  <button onclick="clr()">Clear</button>
  <button onclick="next()">Next &rarr;</button>
  <button class="bo" onclick="previewMask()">Preview Mask</button>
  <button class="bg" onclick="saveAll()">Save All to Python</button>
</div>
<div id="pv" style="margin-top:6px;font-style:italic;color:#666"></div>

<script>
var imgs={_imgs_json};
var names={_names_json};
var dims={_dims_json};
var N=imgs.length;
var allPts={{}};
var ci=0;

function show(i){{
  ci=i;
  document.getElementById('im').src='data:image/jpeg;base64,'+imgs[i];
  document.getElementById('hd').innerHTML=
    '<b>View '+i+'/'+(N-1)+'</b> — '+names[i]+' ('+dims[i][0]+'x'+dims[i][1]+
    ') | Left=<span style="color:green">+pos</span> Right=<span style="color:red">-neg</span>';
  renderPts();
}}

function getC(e){{
  var r=document.getElementById('im').getBoundingClientRect();
  var ow=dims[ci][0],oh=dims[ci][1];
  return[Math.round((e.clientX-r.left)/r.width*ow),
         Math.round((e.clientY-r.top)/r.height*oh)];
}}

document.getElementById('vw').addEventListener('click',function(e){{
  e.preventDefault();var c=getC(e);
  if(!allPts[ci])allPts[ci]=[];
  allPts[ci].push({{x:c[0],y:c[1],l:1}});renderPts();
}});
document.getElementById('vw').addEventListener('contextmenu',function(e){{
  e.preventDefault();var c=getC(e);
  if(!allPts[ci])allPts[ci]=[];
  allPts[ci].push({{x:c[0],y:c[1],l:0}});renderPts();
}});

function renderPts(){{
  var co=document.getElementById('vw');
  co.querySelectorAll('.mk').forEach(m=>m.remove());
  var pts=allPts[ci]||[];
  var r=document.getElementById('im').getBoundingClientRect();
  var ow=dims[ci][0],oh=dims[ci][1];
  pts.forEach(function(p){{
    var m=document.createElement('div');
    m.className='mk '+(p.l===1?'mp':'mn');
    m.style.left=(p.x/ow*r.width)+'px';
    m.style.top=(p.y/oh*r.height)+'px';
    m.textContent=p.l===1?'+':'-';
    co.appendChild(m);
  }});
  var el=document.getElementById('pl');
  if(pts.length===0){{el.innerHTML='No points (text-only for this view)';}}
  else{{el.innerHTML=pts.map(p=>(p.l===1?'<span style="color:green">+</span>':'<span style="color:red">-</span>')+'('+p.x+','+p.y+')').join(' ');}}
  // show count of annotated views
  var n=Object.keys(allPts).filter(k=>allPts[k].length>0).length;
  document.getElementById('hd').innerHTML+=' | <b>'+n+' views annotated</b>';
}}

function undo(){{if(allPts[ci]&&allPts[ci].length)allPts[ci].pop();renderPts();}}
function clr(){{allPts[ci]=[];renderPts();}}
function prev(){{if(ci>0)show(ci-1);}}
function next(){{if(ci<N-1)show(ci+1);}}
function saveAll(){{
  google.colab.kernel.invokeFunction('_receive_all_points',[JSON.stringify(allPts)],{{}});
}}
var _previewing=false;
async function previewMask(){{
  var pts=allPts[ci]||[];
  if(pts.length===0){{document.getElementById('pv').innerHTML='Add points first';return;}}
  if(_previewing)return;
  _previewing=true;
  document.getElementById('pv').innerHTML='Running SAM3 preview...';
  try{{
    var res=await google.colab.kernel.invokeFunction('_preview_mask',[ci,JSON.stringify(pts)],{{}});
    var b64=res.data['text/plain'].replace(/^'/,'').replace(/'$/,'');
    if(b64){{
      document.getElementById('im').src='data:image/jpeg;base64,'+b64;
      document.getElementById('pv').innerHTML='Showing mask overlay. Click Prev/Next to return to original.';
    }} else {{
      document.getElementById('pv').innerHTML='No masks found for this view.';
    }}
  }}catch(e){{
    document.getElementById('pv').innerHTML='Preview error: '+e;
  }}
  _previewing=false;
}}

show(0);
</script>
"""
display(HTML(html))

In [ ]:
# 2.5.5 — Run SAM3 over BOTH view sets -> <folder>/<MASK_NAME>/<stem>.png
# Only segments when masks are missing. Set FORCE_RESEGMENT=True to overwrite.
FORCE_RESEGMENT = False

def _stems(d):
    d = Path(d)
    if not d.exists():
        return set()
    return {p.stem for p in (list(d.glob("*.png")) + list(d.glob("*.jpg")))}

def segment_folder(base_dir, images_dir, per_view_points=None, label=""):
    base_dir, images_dir = Path(base_dir), Path(images_dir)
    mask_dir = base_dir / MASK_NAME
    image_stems = _stems(images_dir)
    mask_stems  = _stems(mask_dir)
    missing     = sorted(image_stems - mask_stems)

    if not FORCE_RESEGMENT and missing == [] and mask_stems:
        print(f"[{label}] all {len(mask_stems)} mask(s) exist in {mask_dir} — skipping SAM3.")
        return {"success_views": len(mask_stems), "total_views": len(image_stems)}

    if mask_stems and not FORCE_RESEGMENT:
        print(f"[{label}] {len(mask_stems)} mask(s) exist; segmenting {len(missing)} missing: {missing}")
        names = missing
    else:
        if FORCE_RESEGMENT:
            print(f"[{label}] FORCE_RESEGMENT=True — (re)segmenting all views.")
        names = None   # segment everything

    pts = {k: v for k, v in (per_view_points or {}).items() if v.get("points")} if per_view_points else {}
    print(f"[{label}] prompt {TEXT_PROMPT!r} -> {MASK_NAME}/"
          + (f" with point prompts for {len(pts)} view(s)" if pts else " (text prompt only)"))

    segmenter = SAM3MultiObjectSegmenter(confidence_threshold=0.1)
    res = segmenter.segment_object_multiview(
        images_dir=images_dir, object_name=MASK_NAME, text_prompt=TEXT_PROMPT,
        output_dir=base_dir, per_view_points=pts or None, image_names=names,
    )
    print(f"[{label}] {res['success_views']}/{res['total_views']} masks -> {mask_dir}")
    return res

# Ground views can use the interactive points from the picker; aerial uses text only.
_pts = globals().get("PER_VIEW_POINTS", {})
segment_folder(GROUND_DIR, IMAGES_DIR, per_view_points=_pts, label="ground")
segment_folder(AERIAL_DIR, AERIAL_IMAGES_DIR, per_view_points=None, label="aerial")

assert len(_stems(GROUND_DIR / MASK_NAME)) > 0, "No ground masks — check the SAM3 prompt/images."
assert len(_stems(AERIAL_DIR / MASK_NAME)) > 0, "No aerial masks — check the SAM3 prompt/images."


## 3 — Load data

Both view sets are loaded with MV-SAM3D's own loader so the format matches exactly
what the Stage-1 / Stage-2 samplers expect (RGB + mask, matched by stem).


In [ ]:
# 3.0 — Load ground (Stage 2) and aerial (Stage 1) views with MV-SAM3D's loader
from load_images_and_masks import load_images_and_masks_from_path
import numpy as np

ground_images, ground_masks, ground_names = load_images_and_masks_from_path(
    GROUND_DIR, mask_prompt=MASK_NAME, image_names=None,
)
aerial_images, aerial_masks, aerial_names = load_images_and_masks_from_path(
    AERIAL_DIR, mask_prompt=MASK_NAME, image_names=None,
)
print(f"Ground (Stage 2) views: {len(ground_images)}  names={ground_names}")
print(f"Aerial (Stage 1) views: {len(aerial_images)}  names={aerial_names}")
print("ground img0", np.array(ground_images[0]).shape, "| aerial img0", np.array(aerial_images[0]).shape)


## 8 — DA3 on ALL views (one shared coordinate frame)

Stage 1 (sparse structure) conditions on per-view **pointmaps** and decodes the
object pose from them; Stage 2's **visibility** weighting needs the ground views'
**camera extrinsics** expressed in that *same* frame. DA3 only produces a consistent
multi-view frame when it sees the images in **one pass**, so we run it **once** on
ground + aerial combined (filenames namespaced so the outputs can be split back),
then route **aerial pointmaps → Stage 1** and **ground extrinsics → Stage 2**.

> Caveat: DA3 pose accuracy on oblique/aerial imagery can be poor. Stage-1 geometry
> and Stage-2 visibility both inherit that frame, so if results look misaligned,
> sanity-check the aerial poses (or fall back to `STAGE2_WEIGHT_SOURCE="entropy"`,
> which needs no extrinsics).


In [ ]:
# 8.0 — Run DA3 ONCE on all views (ground + aerial) so they share one frame
import subprocess, numpy as np, glob, os, shutil
from pathlib import Path

DA3_OUT = OUTPUT_DIR / "da3_all"
DA3_ALL_IMAGES = DA3_OUT / "images"
DA3_ALL_IMAGES.mkdir(parents=True, exist_ok=True)
DA3_NPZ = DA3_OUT / "da3_output.npz"

# Combine both image sets into one dir with namespaced stems, so DA3 treats them as a
# single multi-view capture (-> consistent extrinsics). Prefixes let us split outputs back.
AERIAL_PREFIX, GROUND_PREFIX = "aerial__", "ground__"
_exts = ("*.png", "*.jpg", "*.jpeg", "*.JPG", "*.JPEG", "*.PNG", "*.webp")

def _prefix_copy(src_dir, prefix):
    n = 0
    for e in _exts:
        for p in sorted(Path(src_dir).glob(e)):
            dst = DA3_ALL_IMAGES / f"{prefix}{p.name}"
            if not dst.exists():
                shutil.copy2(p, dst); n += 1
    return n

_na = _prefix_copy(AERIAL_IMAGES_DIR, AERIAL_PREFIX)
_ng = _prefix_copy(IMAGES_DIR, GROUND_PREFIX)
print(f"Combined DA3 input: +{_na} aerial, +{_ng} ground -> {DA3_ALL_IMAGES}")

# Resolve the DA3 weights to the actual snapshot dir (from_pretrained can't load the cache root).
DA3_MODEL_ID = "depth-anything/DA3NESTED-GIANT-LARGE"
_snap_glob = os.path.expanduser(
    "~/.cache/huggingface/hub/models--depth-anything--DA3NESTED-GIANT-LARGE/snapshots/*/")
_snaps = sorted(glob.glob(_snap_glob), key=os.path.getmtime, reverse=True)
DA3_MODEL_PATH = _snaps[0] if _snaps else globals().get("da3_cache", DA3_MODEL_ID)
print("DA3 model path:", DA3_MODEL_PATH)

if not DA3_NPZ.exists():
    cmd = [
        "python", str(MVSAM3D_ROOT / "scripts" / "run_da3.py"),
        "--image_dir", str(DA3_ALL_IMAGES),
        "--output_dir", str(DA3_OUT),
        "--model_path", str(DA3_MODEL_PATH),
        "--process_res", "504",
        "--no_vis",
    ]
    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=str(MVSAM3D_ROOT), capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        print("------ DA3 STDERR ------")
        print(proc.stderr)
        raise RuntimeError(
            f"run_da3.py failed (exit {proc.returncode}). See the DA3 STDERR above for the cause."
        )

da3 = np.load(DA3_NPZ)
da3_pointmaps  = da3["pointmaps_sam3d"]                              # (N, 3, H, W)
da3_extrinsics = da3["extrinsics"] if "extrinsics" in da3 else None  # (N, 3/4, 4) world->camera
da3_stems = [Path(str(p)).stem for p in da3["image_files"]]          # e.g. "aerial__sat1", "ground__0"
print("DA3 pointmaps", da3_pointmaps.shape,
      "| extrinsics", None if da3_extrinsics is None else da3_extrinsics.shape,
      "| views", len(da3_stems))


In [ ]:
# 9.0 — Build the pipeline (same checkpoint layout as mvsam3d_example_current.ipynb)
import os
from inference import Inference   # MV-SAM3D wrapper around the pipeline

# The working notebook exposes pipeline.yaml by symlinking the SAM 3D checkpoint into
# MV-SAM3D/checkpoints/hf, i.e. it runs (after snapshot_download of facebook/sam-3d-objects):
#   !ln -s /content/sam-3d-objects/checkpoints/hf/checkpoints /content/MV-SAM3D/checkpoints/hf
# So run the heavy install + checkpoint download from mvsam3d_example_current.ipynb first.
MODEL_TAG = "hf"
PIPELINE_YAML = MVSAM3D_ROOT / "checkpoints" / MODEL_TAG / "pipeline.yaml"

if not PIPELINE_YAML.exists():
    # Recreate the same symlink if the SAM 3D checkpoint is already downloaded
    src = SAM3D_ROOT / "checkpoints" / "hf" / "checkpoints"
    dst = MVSAM3D_ROOT / "checkpoints" / "hf"
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        os.symlink(src, dst)
        print("Created checkpoint symlink:", dst, "->", src)

assert PIPELINE_YAML.exists(), (
    f"{PIPELINE_YAML} missing. Download the SAM 3D checkpoint and create the "
    f"checkpoints/hf symlink via mvsam3d_example_current.ipynb first."
)

inference = Inference(str(PIPELINE_YAML))
pipeline = inference._pipeline
ss_decoder = pipeline.models["ss_decoder"]      # <- handle used by the guidance
print("Pipeline loaded. ss_decoder:", type(ss_decoder).__name__)


In [ ]:
# 9.2 — preprocess_views: faithful copy of run_multi_view's per-view loop.
# Produces (ss_input_dicts, slat_input_dicts, raw_pointmaps) for ONE view set, so
# we can feed Stage 1 and Stage 2 different sets. Mirrors inference_pipeline.py.
import numpy as np, torch
from PIL import Image

def preprocess_views(pipeline, view_images, view_masks=None, view_pointmaps=None):
    n = len(view_images)
    if view_masks is None:
        view_masks = [None] * n
    if view_pointmaps is None:
        view_pointmaps = [None] * n
    ss_dicts, slat_dicts, raw_pms = [], [], []

    for i, (image, mask, ext_pm) in enumerate(zip(view_images, view_masks, view_pointmaps)):
        # ---- merge mask into the alpha channel (RGBA), exactly like run_multi_view ----
        if mask is not None:
            image = np.array(image)
            mask = np.array(mask)
            if mask.dtype == bool:
                mask = mask.astype(np.uint8) * 255
            elif mask.dtype != np.uint8:
                mask = (mask * 255).astype(np.uint8) if mask.max() <= 1.0 else mask.astype(np.uint8)
            if mask.ndim == 2:
                mask = mask[..., None]
            if image.shape[-1] == 3:
                rgba = np.concatenate([image, mask], axis=-1).astype(np.uint8)
            elif image.shape[-1] == 4:
                rgba = np.concatenate([image[..., :3], mask], axis=-1).astype(np.uint8)
            else:
                raise ValueError(f"Unexpected image shape: {image.shape}")
        else:
            rgba = np.array(image)   # assume already RGBA
        rgba_pil = Image.fromarray(rgba)

        # ---- pointmap + per-stage preprocessing ----
        if hasattr(pipeline, "compute_pointmap"):
            if ext_pm is not None:
                pmd = pipeline.compute_pointmap(rgba_pil, pointmap=torch.from_numpy(ext_pm).float())
            else:
                pmd = pipeline.compute_pointmap(rgba_pil, pointmap=None)
            pointmap = pmd["pointmap"]
            if pointmap is not None:
                pmm = pointmap.detach()
                if hasattr(type(pipeline), "_down_sample_img"):
                    pmm = type(pipeline)._down_sample_img(pmm)
                raw_pms.append(pmm.cpu().permute(1, 2, 0).numpy())
            ss_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.ss_preprocessor, pointmap=pointmap))
            slat_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.slat_preprocessor))
        else:
            ss_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.ss_preprocessor))
            slat_dicts.append(pipeline.preprocess_image(rgba_pil, pipeline.slat_preprocessor))

    return ss_dicts, slat_dicts, raw_pms


def run_split_staging(pipeline, *, stage1_images, stage1_masks, stage1_pointmaps,
                      stage2_images, stage2_masks,
                      seed=0, stage1_steps=25, stage2_steps=25, mode="multidiffusion",
                      decode_formats=("mesh", "gaussian"),
                      ss_weighting=True, ss_entropy_layer=9, ss_entropy_alpha=60.0, ss_warmup_steps=1,
                      with_mesh_postprocess=False, with_texture_baking=False, use_vertex_color=True,
                      stage1_only=False, weighting_config=None):
    """Stage 1 on stage1_* (aerial); Stage 2 (SLAT) on stage2_* (ground) at the
    Stage-1 coords. Faithful to run_multi_view, just split across two view sets."""
    if seed is not None:
        torch.manual_seed(seed)

    # ---- Stage 1: AERIAL -> coarse sparse structure (coords) ----
    print(f"[Stage 1] preprocessing {len(stage1_images)} AERIAL view(s)...")
    ss_dicts, _, raw_pms = preprocess_views(pipeline, stage1_images, stage1_masks, stage1_pointmaps)
    print("[Stage 1] sampling sparse structure...")
    ss_return = pipeline.sample_sparse_structure_multi_view(
        ss_dicts, inference_steps=stage1_steps, mode=mode,
        ss_weighting=ss_weighting, ss_entropy_layer=ss_entropy_layer,
        ss_entropy_alpha=ss_entropy_alpha, ss_warmup_steps=ss_warmup_steps,
    )
    pm_scale = ss_dicts[0].get("pointmap_scale", None)
    pm_shift = ss_dicts[0].get("pointmap_shift", None)
    ss_return.update(pipeline.pose_decoder(ss_return, scene_scale=pm_scale, scene_shift=pm_shift))
    ss_return["pointmap_scale"] = pm_scale
    ss_return["pointmap_shift"] = pm_shift
    if "all_view_poses_raw" in ss_return and hasattr(pipeline, "_decode_all_view_poses"):
        ss_return["all_view_poses_decoded"] = pipeline._decode_all_view_poses(
            ss_return["all_view_poses_raw"], ss_dicts)
    if "scale" in ss_return:
        ss_return["scale"] = ss_return["scale"] * ss_return["downsample_factor"]
    coords = ss_return["coords"]
    print(f"[Stage 1] coords: {tuple(coords.shape)}")

    if stage1_only:
        ss_return["voxel"] = ss_return["coords"][:, 1:] / 64 - 0.5
        print("[Stage 1] stage1_only=True -> returning sparse structure (no Stage 2).")
        return ss_return

    # ---- Stage 2: GROUND -> SLAT on the Stage-1 coords ----
    print(f"[Stage 2] preprocessing {len(stage2_images)} GROUND view(s)...")
    _, slat_dicts, _ = preprocess_views(pipeline, stage2_images, stage2_masks, None)
    print("[Stage 2] sampling structured latent...")
    if weighting_config is not None:
        # visibility/mixed needs the Stage-1 object pose to place ground cameras in canonical space
        if getattr(weighting_config, "weight_source", "entropy") in ("visibility", "mixed"):
            weighting_config.object_pose = {
                "scale": ss_return.get("scale"),
                "rotation": ss_return.get("rotation"),
                "translation": ss_return.get("translation"),
            }
        slat, _ = pipeline.sample_slat_multi_view_weighted(
            slat_dicts, coords, inference_steps=stage2_steps, weighting_config=weighting_config)
    else:
        slat = pipeline.sample_slat_multi_view(slat_dicts, coords, inference_steps=stage2_steps, mode=mode)

    outputs = pipeline.decode_slat(slat, list(decode_formats))
    outputs = pipeline.postprocess_slat_output(
        outputs, with_mesh_postprocess, with_texture_baking, use_vertex_color)
    result = {**ss_return, **outputs, "view_ss_input_dicts": ss_dicts}
    if raw_pms:
        result["raw_view_pointmaps"] = raw_pms
    print("Finished split-staging run.")
    return result

print("preprocess_views + run_split_staging ready.")


## 4 — Merge aerial + ground into ONE multi-view set

The base model is meant to consume the whole capture at once, so we concatenate the aerial and
ground views into a single set and reuse the **one shared DA3 frame** computed above (§8). Every
view carries its DA3 pointmap; Stage 1 and Stage 2 both run on this merged set.


In [ ]:
# 4.0 — Merge views + match the shared-frame DA3 pointmaps (by namespaced stem)
import numpy as np, matplotlib.pyplot as plt

all_images = list(aerial_images) + list(ground_images)
all_masks  = list(aerial_masks)  + list(ground_masks)
all_names  = [f"{AERIAL_PREFIX}{n}" for n in aerial_names] + [f"{GROUND_PREFIX}{n}" for n in ground_names]

_da3_idx = {st: i for i, st in enumerate(da3_stems)}
all_pointmaps = []
for nm in all_names:
    if nm in _da3_idx:
        all_pointmaps.append(da3_pointmaps[_da3_idx[nm]])
    else:
        print(f"  WARN: no DA3 pointmap for {nm}; pipeline will self-compute one")
        all_pointmaps.append(None)

N_VIEWS = len(all_images)
print(f"Merged multi-view set: {N_VIEWS} views "
      f"({len(aerial_images)} aerial + {len(ground_images)} ground); "
      f"pointmaps matched: {sum(p is not None for p in all_pointmaps)}/{N_VIEWS}")

# quick montage so we can eyeball the exact dataset the model will ingest
_n = N_VIEWS; _cols = min(6, _n); _rows = (_n + _cols - 1) // _cols
fig, axs = plt.subplots(_rows, _cols, figsize=(2.4 * _cols, 2.4 * _rows))
axs = np.atleast_1d(axs).ravel()
for i in range(len(axs)):
    axs[i].axis("off")
    if i < _n:
        im = np.array(all_images[i])
        axs[i].imshow(im)
        tag = "aerial" if i < len(aerial_images) else "ground"
        axs[i].set_title(f"{tag} · {all_names[i].split('__',1)[-1]}", fontsize=8)
plt.suptitle(f"Merged input set ({N_VIEWS} views) fed to base MV-SAM3D"); plt.tight_layout(); plt.show()


In [ ]:
# 4.1 — Preprocess ALL views once (shared by every metric) + locate the SS VAE encoder
import torch, numpy as np

ss_generator   = pipeline.models["ss_generator"]
slat_generator = pipeline.models["slat_generator"]
ss_decoder     = pipeline.models["ss_decoder"]
ss_encoder     = pipeline.models["ss_encoder"] if "ss_encoder" in pipeline.models else None
print("SS VAE encoder available:", ss_encoder is not None,
      "" if ss_encoder is None else f"({type(ss_encoder).__name__}) -> round-trip u_r enabled")
if ss_encoder is None:
    print("  NOTE: this checkpoint did not ship ss_encoder weights; u_r will be skipped "
          "(u_s and u_f do not need it).")

# preprocess every view for BOTH stages once; reused by baseline + all metric passes
ss_dicts_all, slat_dicts_all, _raw_pms_all = preprocess_views(
    pipeline, all_images, all_masks, all_pointmaps)
print(f"Preprocessed {len(ss_dicts_all)} views for SS and {len(slat_dicts_all)} for SLAT.")

_DEV = "cuda" if torch.cuda.is_available() else "cpu"

def _seed_all(s):
    """Seed the GLOBAL torch RNG (the flow sampler draws its initial noise from it)."""
    torch.manual_seed(int(s))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(s))

def ss_latent_to_occupancy(shape_latent, as_prob=True):
    """(B,4096,8) generator latent -> occupancy grid, mirroring sample_sparse_structure."""
    B = shape_latent.shape[0]
    z_cube = shape_latent.permute(0, 2, 1).contiguous().view(B, 8, 16, 16, 16).to(_DEV)
    with torch.no_grad(), torch.autocast(device_type=_DEV, enabled=(_DEV == "cuda")):
        logits = ss_decoder(z_cube)                       # (B,1,64,64,64)
    logits = logits.float()
    return torch.sigmoid(logits) if as_prob else (logits > 0).float()

print("Helpers ready: _seed_all(), ss_latent_to_occupancy().")


## 5 — Baseline reconstruction (base MV-SAM3D, both stages, all views)

One deterministic reference run at `SEED`. We keep the Stage-1 latent (`ẑ_SS`) and the SS coords
(the fixed geometry Stage-2 latents live on), then preview the `64³` occupancy — this is the
"after Stage 1" geometry the metrics analyze.


In [ ]:
# 5.0 — Baseline: run BOTH stages once on the merged view set
import torch, numpy as np, matplotlib.pyplot as plt

_seed_all(SEED)
baseline = run_split_staging(
    pipeline,
    stage1_images=all_images, stage1_masks=all_masks, stage1_pointmaps=all_pointmaps,
    stage2_images=all_images, stage2_masks=all_masks,
    seed=SEED, stage1_steps=STAGE1_STEPS, stage2_steps=STAGE2_STEPS, mode=MODE,
    decode_formats=("mesh", "gaussian"),
    ss_weighting=SS_WEIGHTING, ss_entropy_alpha=SS_ENTROPY_ALPHA, ss_warmup_steps=SS_WARMUP_STEPS,
    weighting_config=None,   # base SLAT = plain multidiffusion (no visibility weighting)
)
baseline_shape  = baseline["shape"].detach()             # (B,4096,8) SS latent (ẑ_SS)
baseline_coords = baseline["coords"]                     # (M,4) SS voxel coords
baseline_occ    = ss_latent_to_occupancy(baseline_shape, as_prob=True)[0, 0].cpu().numpy()
print("SS latent:", tuple(baseline_shape.shape),
      "| coords:", tuple(baseline_coords.shape),
      "| occupancy:", baseline_occ.shape, "| occupied(>0.5):", int((baseline_occ > 0.5).sum()))

# preview the baseline Stage-1 occupancy (max projection along each axis)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, axis, lbl in zip(axs, (0, 1, 2), ("axis-0 (i)", "axis-1 (j)", "axis-2 (k)")):
    ax.imshow(baseline_occ.max(axis=axis), origin="lower", cmap="viridis")
    ax.set_title(f"max-proj {lbl}"); ax.axis("off")
plt.suptitle("Baseline Stage-1 occupancy (64³) — the geometry the metrics probe")
plt.tight_layout(); plt.savefig(HALLUC_OUT / "baseline_occupancy.png", dpi=130); plt.show()

np.save(HALLUC_OUT / "baseline_occupancy.npy", baseline_occ)
print("Saved baseline occupancy to", HALLUC_OUT)


## 6 — Metric 1: tokenizer round-trip residual `u_r`  (Stage 1 / SS)

`u_r = ‖ẑ − E_S(D_S(ẑ))‖`. Decode the generated SS latent to `64³` occupancy, re-encode it with
the SS VAE encoder, and measure the per-voxel latent residual. **Low** = the latent lies on the
VAE manifold (trustworthy); **high** = it drifts off-manifold (hallucination-prone). We contrast
against a pure-noise latent, which should *not* survive the round trip.


In [ ]:
# 6.0 — u_r: SS latent round-trip residual
import torch, numpy as np, matplotlib.pyplot as plt

def ss_round_trip(shape_latent):
    """z (B,4096,8) -> z_cube -> D_S -> occupancy -> E_S -> z' ; return (z_cube, z', resid_map)."""
    B = shape_latent.shape[0]
    z_cube = shape_latent.permute(0, 2, 1).contiguous().view(B, 8, 16, 16, 16).to(_DEV).float()
    with torch.no_grad(), torch.autocast(device_type=_DEV, enabled=(_DEV == "cuda")):
        occ_logits = ss_decoder(z_cube)                 # (B,1,64,64,64)
        occ_bin = (occ_logits > 0).float()              # binary occupancy = encoder's training input
        z_re = ss_encoder(occ_bin, sample_posterior=False)   # (B,8,16,16,16) posterior mean
    z_re = z_re.float()
    resid_map = (z_cube - z_re).pow(2).mean(dim=1).sqrt()    # (B,16,16,16) per-voxel RMS over channels
    return z_cube, z_re, resid_map

u_r = u_r_rand = None
if ss_encoder is None:
    print("ss_encoder unavailable -> skipping u_r (round-trip needs the VAE encoder).")
else:
    try:
        _, _, ur_map = ss_round_trip(baseline_shape)
        u_r = float(ur_map.mean())
        torch.manual_seed(12345)
        z_rand = torch.randn_like(baseline_shape)
        _, _, ur_rand_map = ss_round_trip(z_rand)
        u_r_rand = float(ur_rand_map.mean())
        print(f"u_r  (generated SS latent)    = {u_r:.4f}")
        print(f"u_r  (random-noise reference) = {u_r_rand:.4f}   (off-manifold upper reference)")
        print(f"ratio generated/random        = {u_r / max(u_r_rand, 1e-8):.3f}  (closer to 0 = safer)")

        m = ur_map[0].cpu().numpy()
        fig, axs = plt.subplots(1, 4, figsize=(16, 4))
        for ax, axis, lbl in zip(axs[:3], (0, 1, 2), ("i", "j", "k")):
            ax.imshow(m.max(axis=axis), origin="lower", cmap="inferno")
            ax.set_title(f"u_r max-proj {lbl}"); ax.axis("off")
        axs[3].hist(m.reshape(-1), bins=40, color="crimson", alpha=.8)
        axs[3].axvline(u_r, color="k", ls="--", label=f"gen mean={u_r:.3f}")
        axs[3].axvline(u_r_rand, color="gray", ls=":", label=f"rand mean={u_r_rand:.3f}")
        axs[3].set_title("per-voxel u_r (16³)"); axs[3].legend(fontsize=8)
        plt.suptitle("Metric 1 — SS latent round-trip residual (higher = more off-manifold / hallucinated)")
        plt.tight_layout(); plt.savefig(HALLUC_OUT / "u_r_roundtrip.png", dpi=130); plt.show()
    except Exception as e:
        print("u_r computation failed (encoder input format?):", repr(e))
        u_r = u_r_rand = None


### 6.5 — Qualitative: paint `u_r` onto the geometry

Note: `ẑ` here is the **generated** latent (the denoised Stage-1 output that decodes to the building)
— *not* noise. Only the reference `u_r_rand` uses pure noise, and we never paint that. So the surface
below is the real reconstructed building; we just overlay the round-trip residual as color.

The residual lives in the `16³` latent grid. We upsample it (nearest) to the `64³` occupancy grid to
paint it on the surface — this is an **approximate localization**: the SS decoder upsamples 16→64 by
4×, so each latent cell `(i,j,k)` maps roughly to the `4³` occupancy block around it (conv receptive
fields blur this a little). Bright = that region's latent code is off-manifold. We also decode the
occupancy **before** (`ẑ`) and **after** (`E_S(D_S(ẑ))`) the round trip and show their difference —
the regions the encoder "couldn't put back" are the hallucinated ones. Exports a vertex-colored `.glb`.

In [ ]:
# 6.5 — Paint the round-trip residual u_r onto the surface (per voxel)
import numpy as np, matplotlib.pyplot as plt, matplotlib.cm as cm, trimesh, torch
from skimage import measure

if ss_encoder is None or "ur_map" not in globals():
    print("u_r unavailable (no SS encoder) -> skipping the round-trip painting.")
else:
    z_cube, z_re, ur_map_q = ss_round_trip(baseline_shape)
    with torch.no_grad(), torch.autocast(device_type=_DEV, enabled=(_DEV == "cuda")):
        occ_from_z   = torch.sigmoid(ss_decoder(z_cube)).float()[0, 0].cpu().numpy()   # occupancy from ẑ
        occ_from_zre = torch.sigmoid(ss_decoder(z_re)).float()[0, 0].cpu().numpy()     # from E_S(D_S(ẑ))
    # upsample the 16³ residual to the 64³ grid (nearest) so it can be painted on the surface
    ur16 = ur_map_q[0].cpu().numpy()
    ur64 = np.repeat(np.repeat(np.repeat(ur16, 4, 0), 4, 1), 4, 2)

    _lvl = 0.5 if occ_from_z.max() > 0.5 else float(occ_from_z.max()) * 0.5
    verts, faces, normals, _ = measure.marching_cubes(occ_from_z, level=_lvl)
    vi = np.clip(np.round(verts).astype(int), 0, 63)
    vert_ur = ur64[vi[:, 0], vi[:, 1], vi[:, 2]]
    vmax = float(np.percentile(vert_ur, 99)); vmax = vmax if vmax > 1e-6 else 1.0
    vmesh = verts / 64.0 - 0.5
    tri = trimesh.Trimesh(vertices=vmesh, faces=faces, vertex_normals=normals,
                          vertex_colors=(cm.inferno(np.clip(vert_ur / vmax, 0, 1)) * 255).astype(np.uint8),
                          process=False)
    glb = HALLUC_OUT / "u_r_painted_mesh.glb"; tri.export(glb)
    print(f"u_r-painted mesh saved: {glb.name} | residual range [{vert_ur.min():.3f}, {vert_ur.max():.3f}]")

    # Fig 1 — surface painted by u_r (3 angles + shared colorbar)
    face_ur = vert_ur[faces].mean(1)
    fig = plt.figure(figsize=(15, 5)); last = None
    for i, (az, el) in enumerate([(-60, 25), (30, 25), (90, 80)]):
        ax = fig.add_subplot(1, 3, i + 1, projection="3d")
        last = ax.plot_trisurf(vmesh[:, 0], vmesh[:, 1], vmesh[:, 2], triangles=faces,
                               cmap="inferno", linewidth=0, antialiased=False, shade=False)
        last.set_array(face_ur); last.set_clim(0, vmax)
        ax.view_init(elev=el, azim=az); ax.set_box_aspect((1, 1, 1))
        ax.set_title(f"u_r painted ({az},{el})", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    fig.colorbar(last, ax=fig.axes, shrink=0.5, label="round-trip residual u_r")
    plt.suptitle("Metric 1 (qualitative) — round-trip residual on the surface (bright = off-manifold / hallucinated)")
    plt.savefig(HALLUC_OUT / "u_r_painted.png", dpi=130, bbox_inches="tight"); plt.show()

    # Fig 2 — occupancy before vs after the round trip, and where it changed
    fig2, axs = plt.subplots(1, 3, figsize=(13, 4))
    for ax, volm, ttl in zip(axs, [occ_from_z, occ_from_zre, np.abs(occ_from_z - occ_from_zre)],
                             ["occ(ẑ)", "occ(E_S∘D_S(ẑ))", "|difference|"]):
        ax.imshow(volm.max(axis=2), origin="lower", cmap="inferno"); ax.set_title(ttl); ax.axis("off")
    plt.suptitle("Round trip in occupancy space (top-down max projection)")
    plt.tight_layout(); plt.savefig(HALLUC_OUT / "u_r_roundtrip_occupancy.png", dpi=130); plt.show()

## 7 — Metric 2: inter-seed variance `u_s`  (Stage 1 + Stage 2)

Re-run the sampler with different **initial random latents** and measure how much the output
disperses. Concentrated predictions → a well-determined transition; dispersed ones → where the
model diverges. Two **aligned** per-voxel measures (each needs a common grid, so each isolates one stage):

- **Stage 1 — shape uncertainty:** vary the SS seed → per-voxel **std of the `64³` occupancy** across seeds.
- **Stage 2 — appearance uncertainty:** freeze the Stage-1 coords, vary only the SLAT seed → per-voxel
  **L2 dispersion of the 8-D SLAT feature** (`√Σ_c var_c` = RMS distance of each seed's feature vector from
  the per-voxel mean). Features stay aligned because the coords are fixed.

You can't collapse *both stages varying together* into one per-voxel number (each seed then has a different
voxel grid), so the **end-to-end** view lives in §7.9 as (a) a per-seed gallery of full Stage-1+Stage-2
reconstructions and (b) two uncertainty-painted meshes (shape from Stage 1, appearance from Stage 2).


In [ ]:
# 7.0 — u_s (Stage 1 / SS): vary the initial latent, measure per-voxel occupancy dispersion
import torch, numpy as np, matplotlib.pyplot as plt

occ_seeds, coords_seeds = [], []
for s in SEED_LIST:
    _seed_all(s)
    ssr = pipeline.sample_sparse_structure_multi_view(
        ss_dicts_all, inference_steps=STAGE1_STEPS, mode=MODE,
        ss_weighting=SS_WEIGHTING, ss_entropy_alpha=SS_ENTROPY_ALPHA, ss_warmup_steps=SS_WARMUP_STEPS)
    occ = ss_latent_to_occupancy(ssr["shape"], as_prob=True)[0, 0].cpu().numpy()
    occ_seeds.append(occ)
    coords_seeds.append(ssr["coords"])          # keep each seed's Stage-1 coords for the end-to-end gallery (§7.9c)
    print(f"  seed {s}: occupied(>0.5) voxels = {int((occ > 0.5).sum())}")

occ_stack = np.stack(occ_seeds, 0)          # (S,64,64,64)
occ_mean  = occ_stack.mean(0)
us_ss_map = occ_stack.std(0)                # per-voxel std across seeds
_active   = occ_mean > 0.05                 # dynamism mask: only where the model puts mass
u_s_ss     = float(us_ss_map[_active].mean()) if _active.any() else float(us_ss_map.mean())
u_s_ss_raw = float(us_ss_map.mean())
print(f"u_s (SS, active-region mean std) = {u_s_ss:.4f}   | whole-grid mean std = {u_s_ss_raw:.4f}")

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, axis, lbl in zip(axs[:3], (0, 1, 2), ("i", "j", "k")):
    ax.imshow(us_ss_map.max(axis=axis), origin="lower", cmap="magma")
    ax.set_title(f"seed-std max-proj {lbl}"); ax.axis("off")
axs[3].hist(us_ss_map[_active].reshape(-1), bins=40, color="darkorange", alpha=.85)
axs[3].axvline(u_s_ss, color="k", ls="--", label=f"active mean={u_s_ss:.3f}")
axs[3].set_title("per-voxel occupancy std (active)"); axs[3].legend(fontsize=8)
plt.suptitle(f"Metric 2 — Stage-1 inter-seed variance across {len(SEED_LIST)} seeds "
             f"(bright = geometry the model is unsure about)")
plt.tight_layout(); plt.savefig(HALLUC_OUT / "u_s_ss.png", dpi=130); plt.show()
np.save(HALLUC_OUT / "u_s_ss_map.npy", us_ss_map)


In [ ]:
# 7.5 — u_s (Stage 2 / SLAT): freeze Stage-1 coords, vary only the SLAT seed
import torch, numpy as np, matplotlib.pyplot as plt

u_s_slat = None
if not RUN_SLAT_METRICS:
    print("RUN_SLAT_METRICS=False -> skipping SLAT inter-seed variance.")
else:
    coords_fixed = baseline_coords   # freeze geometry so features align voxel-for-voxel across seeds
    feats_seeds = []
    for s in SEED_LIST:
        _seed_all(s)
        slat = pipeline.sample_slat_multi_view(
            slat_dicts_all, coords_fixed, inference_steps=STAGE2_STEPS, mode=MODE)
        feats_seeds.append(slat.feats.detach().float().cpu().numpy())   # (M,8) aligned to coords_fixed
    feat_stack = np.stack(feats_seeds, 0)              # (S,M,8)
    us_slat_vox = np.sqrt(feat_stack.var(0).sum(-1))   # (M,) per-voxel L2 dispersion of the 8-D feature vector
    u_s_slat = float(us_slat_vox.mean())
    print(f"u_s (SLAT, per-voxel L2 feature dispersion) = {u_s_slat:.4f}  over {us_slat_vox.shape[0]} voxels")

    cc = coords_fixed[:, 1:].detach().cpu().numpy()
    fig = plt.figure(figsize=(13, 4))
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    p = ax.scatter(cc[:, 0], cc[:, 1], cc[:, 2], c=us_slat_vox, cmap="magma", s=4)
    fig.colorbar(p, ax=ax, shrink=.6, label="feature L2 dispersion")
    ax.set_title("SLAT inter-seed variance (per voxel)")
    ax.set_xlabel("i"); ax.set_ylabel("j"); ax.set_zlabel("k")
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.hist(us_slat_vox, bins=40, color="purple", alpha=.85)
    ax2.axvline(u_s_slat, color="k", ls="--", label=f"mean={u_s_slat:.3f}")
    ax2.set_title("per-voxel SLAT feature L2 dispersion"); ax2.legend(fontsize=8)
    plt.suptitle(f"Metric 2 — Stage-2 (SLAT) appearance uncertainty across {len(SEED_LIST)} seeds "
                 f"(Stage-1 geometry frozen)")
    plt.tight_layout(); plt.savefig(HALLUC_OUT / "u_s_slat.png", dpi=130); plt.show()


### 7.9 — Qualitative: *see* the seed disagreement (per stage + end-to-end)

The `u_s` numbers above collapse per-voxel variance into a scalar. Here we make it visual, at two levels:

**Per-stage (aligned, reuse §7.0/§7.5 — no extra model runs):**
- **7.9a** — per-seed gallery of the **Stage-1 geometry** (which structures appear/vanish as the seed changes).
- **7.9b** — **shape-uncertainty mesh**: the mean surface painted by per-voxel occupancy std (Stage-1), exported
  as a vertex-colored `.glb` you can rotate.

**End-to-end (both stages vary together — the full reconstruction):**
- **7.9c** — per-seed gallery of the **final Stage-1+Stage-2 reconstruction**. Reuses each seed's stored
  Stage-1 coords and runs Stage 2 + mesh-decode per seed (≈ `len(SEED_LIST)` extra SLAT runs + decodes).
- **7.9d** — **appearance-uncertainty mesh**: the baseline surface painted by the per-voxel SLAT L2 feature
  dispersion from §7.5 (how much Stage 2 disagrees on the fixed shape), exported as a `.glb`.

Together: 7.9b = "how much does the *shape* wobble", 7.9d = "how much does the *stage-2 refinement* wobble",
7.9c = "what do the fully-varied reconstructions actually look like".

In [ ]:
# 7.9a — Per-seed Stage-1 geometry gallery (reuses occ_seeds from §7.0)
import numpy as np, matplotlib.pyplot as plt
try:
    from skimage import measure
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-image"], check=True)
    from skimage import measure

_S = len(occ_seeds)
_cols = min(_S, 4); _rows = (_S + _cols - 1) // _cols
fig = plt.figure(figsize=(4 * _cols, 4 * _rows))
for i, occ in enumerate(occ_seeds):
    ax = fig.add_subplot(_rows, _cols, i + 1, projection="3d")
    try:
        v, f, n, _ = measure.marching_cubes(occ, level=0.5)
        v = v / 64.0 - 0.5
        ax.plot_trisurf(v[:, 0], v[:, 1], v[:, 2], triangles=f, cmap="viridis",
                        linewidth=0, antialiased=False)
    except Exception as e:
        ax.text2D(0.5, 0.5, f"empty\n{e}", ha="center", transform=ax.transAxes)
    ax.view_init(elev=25, azim=-60); ax.set_box_aspect((1, 1, 1))
    ax.set_title(f"seed {SEED_LIST[i]}  ({int((occ > 0.5).sum())} vox)", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
plt.suptitle("Stage-1 geometry per initial-latent seed (differences = model uncertainty)")
plt.tight_layout(); plt.savefig(HALLUC_OUT / "u_s_seed_gallery.png", dpi=130); plt.show()

In [ ]:
# 7.9b — Uncertainty-painted mesh: baseline surface colored by inter-seed std (u_s map)
import numpy as np, matplotlib.pyplot as plt, matplotlib.cm as cm, trimesh
from skimage import measure

_lvl = 0.5 if occ_mean.max() > 0.5 else float(occ_mean.max()) * 0.5
try:
    verts, faces, normals, _ = measure.marching_cubes(occ_mean, level=_lvl)
    vi = np.clip(np.round(verts).astype(int), 0, 63)                 # voxel index per vertex
    vert_unc = us_ss_map[vi[:, 0], vi[:, 1], vi[:, 2]]               # inter-seed std at each vertex
    vmax = float(np.percentile(vert_unc, 99)); vmax = vmax if vmax > 1e-6 else 1.0
    vcol = (cm.magma(np.clip(vert_unc / vmax, 0, 1)) * 255).astype(np.uint8)   # (V,4) RGBA

    vmesh = verts / 64.0 - 0.5
    mesh = trimesh.Trimesh(vertices=vmesh, faces=faces, vertex_normals=normals,
                           vertex_colors=vcol, process=False)
    glb_path = HALLUC_OUT / "uncertainty_painted_mesh.glb"; mesh.export(glb_path)
    print(f"Painted mesh: {len(verts)} verts | seed-std range "
          f"[{vert_unc.min():.3f}, {vert_unc.max():.3f}] | saved {glb_path.name}")

    face_unc = vert_unc[faces].mean(1)                              # per-face uncertainty
    fig = plt.figure(figsize=(15, 5)); last = None
    for i, (az, el) in enumerate([(-60, 25), (30, 25), (90, 80)]):
        ax = fig.add_subplot(1, 3, i + 1, projection="3d")
        last = ax.plot_trisurf(vmesh[:, 0], vmesh[:, 1], vmesh[:, 2], triangles=faces,
                               cmap="magma", linewidth=0, antialiased=False, shade=False)
        last.set_array(face_unc); last.set_clim(0, vmax)
        ax.view_init(elev=el, azim=az); ax.set_box_aspect((1, 1, 1))
        ax.set_title(f"painted ({az},{el})", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    fig.colorbar(last, ax=fig.axes, shrink=0.5, label="inter-seed std (hallucination-prone →)")
    plt.suptitle("Baseline Stage-1 mesh painted by inter-seed uncertainty")
    plt.savefig(HALLUC_OUT / "uncertainty_painted_mesh.png", dpi=130, bbox_inches="tight"); plt.show()
except Exception as e:
    print("Uncertainty-painted mesh failed (degenerate occupancy?):", repr(e))

In [ ]:
# 7.9c — End-to-end per-seed FINAL reconstruction gallery (Stage 1 + Stage 2 BOTH vary)
# Reuses each seed's stored Stage-1 coords (§7.0), runs Stage 2 + mesh-decode per seed.
import numpy as np, matplotlib.pyplot as plt, torch, trimesh

if not RUN_SLAT_METRICS:
    print("RUN_SLAT_METRICS=False -> skipping end-to-end reconstruction gallery.")
else:
    N_E2E_SHOW = min(len(SEED_LIST), 8)     # decode/plot this many seeds (raise it; costs 1 SLAT + 1 decode each)
    e2e = []                                 # (seed, verts, faces, vertex_rgb)
    for i in range(N_E2E_SHOW):
        s = SEED_LIST[i]
        try:
            _seed_all(s)                     # same seed feeds Stage 2 (Stage-1 coords already came from this seed)
            slat = pipeline.sample_slat_multi_view(
                slat_dicts_all, coords_seeds[i], inference_steps=STAGE2_STEPS, mode=MODE)
            m = pipeline.decode_slat(slat, ["mesh"])["mesh"][0]
            v = m.vertices.float().detach().cpu().numpy()
            f = m.faces.detach().cpu().numpy().astype(np.int64)
            vc = None
            if getattr(m, "vertex_attrs", None) is not None:
                vc = np.clip(m.vertex_attrs[:, :3].float().detach().cpu().numpy(), 0, 1)
            e2e.append((s, v, f, vc))
            rgba = (np.concatenate([vc, np.ones((len(vc), 1))], 1) * 255).astype(np.uint8) if vc is not None else None
            trimesh.Trimesh(vertices=v, faces=f, vertex_colors=rgba, process=False).export(
                HALLUC_OUT / f"recon_seed{s}.glb")
            print(f"  seed {s}: final mesh {v.shape[0]} verts / {f.shape[0]} faces -> recon_seed{s}.glb")
        except Exception as e:
            print(f"  seed {s}: end-to-end decode failed -> {e!r}")
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()       # free decoder memory between seeds

    if e2e:
        _cols = min(len(e2e), 4); _rows = (len(e2e) + _cols - 1) // _cols
        fig = plt.figure(figsize=(4 * _cols, 4 * _rows))
        for j, (s, v, f, vc) in enumerate(e2e):
            ax = fig.add_subplot(_rows, _cols, j + 1, projection="3d")
            ax.plot_trisurf(v[:, 0], v[:, 1], v[:, 2], triangles=f, cmap="viridis",
                            linewidth=0, antialiased=False)
            ax.view_init(elev=25, azim=-60); ax.set_box_aspect((1, 1, 1))
            ax.set_title(f"seed {s}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        plt.suptitle(f"End-to-end per-seed FINAL reconstruction (Stage 1 + Stage 2 both vary; "
                     f"{len(e2e)}/{len(SEED_LIST)} seeds shown)")
        plt.tight_layout(); plt.savefig(HALLUC_OUT / "u_s_e2e_gallery.png", dpi=130); plt.show()
        print(f"Saved {len(e2e)} per-seed .glb reconstructions in {HALLUC_OUT} (open in any 3D viewer).")

In [ ]:
# 7.9d — Appearance-uncertainty mesh: baseline surface painted by SLAT L2 feature dispersion (from §7.5)
import numpy as np, matplotlib.pyplot as plt, matplotlib.cm as cm, trimesh
from skimage import measure

if not RUN_SLAT_METRICS or u_s_slat is None:
    print("RUN_SLAT_METRICS=False (or no SLAT variance) -> skipping appearance-uncertainty mesh.")
else:
    # scatter per-voxel SLAT dispersion (on the frozen baseline coords) into a 64^3 grid
    ci = coords_fixed[:, 1:].detach().cpu().numpy().astype(int)
    slat_grid = np.zeros((64, 64, 64), np.float32)
    slat_grid[ci[:, 0], ci[:, 1], ci[:, 2]] = us_slat_vox
    _lvl = 0.5 if baseline_occ.max() > 0.5 else float(baseline_occ.max()) * 0.5
    try:
        verts, faces, normals, _ = measure.marching_cubes(baseline_occ, level=_lvl)
        vi = np.clip(np.round(verts).astype(int), 0, 63)
        vdisp = slat_grid[vi[:, 0], vi[:, 1], vi[:, 2]]                 # SLAT dispersion at each vertex
        pos = vdisp[vdisp > 0]
        vmax = float(np.percentile(pos, 99)) if pos.size else 1.0
        vmax = vmax if vmax > 1e-6 else 1.0
        vcol = (cm.magma(np.clip(vdisp / vmax, 0, 1)) * 255).astype(np.uint8)

        vmesh = verts / 64.0 - 0.5
        trimesh.Trimesh(vertices=vmesh, faces=faces, vertex_normals=normals,
                        vertex_colors=vcol, process=False).export(HALLUC_OUT / "appearance_uncertainty_mesh.glb")
        print(f"Appearance mesh: {len(verts)} verts | SLAT dispersion range "
              f"[{vdisp.min():.3f}, {vdisp.max():.3f}] | saved appearance_uncertainty_mesh.glb")

        face_disp = vdisp[faces].mean(1)
        fig = plt.figure(figsize=(15, 5)); last = None
        for i, (az, el) in enumerate([(-60, 25), (30, 25), (90, 80)]):
            ax = fig.add_subplot(1, 3, i + 1, projection="3d")
            last = ax.plot_trisurf(vmesh[:, 0], vmesh[:, 1], vmesh[:, 2], triangles=faces,
                                   cmap="magma", linewidth=0, antialiased=False, shade=False)
            last.set_array(face_disp); last.set_clim(0, vmax)
            ax.view_init(elev=el, azim=az); ax.set_box_aspect((1, 1, 1))
            ax.set_title(f"appearance uncertainty ({az},{el})", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        fig.colorbar(last, ax=fig.axes, shrink=0.5, label="SLAT feature L2 dispersion (Stage-2 disagreement →)")
        plt.suptitle("Stage-2 (SLAT) appearance uncertainty painted on the baseline surface")
        plt.savefig(HALLUC_OUT / "appearance_uncertainty_mesh.png", dpi=130, bbox_inches="tight"); plt.show()
    except Exception as e:
        print("Appearance-uncertainty mesh failed (degenerate occupancy?):", repr(e))

## 8 — Metric 3: flow instability `u_f`  (Stage 1 + Stage 2)

For rectified flow, the sampler follows `x_{t+dt} = x_t + v(x_t,t)·dt` with `t:0→1` (noise→data).
The **clean-data prediction** at any point is `x̂₁ = x_t + (1−t)·v`. We wrap the ODE **solver** to log
every Euler step of a single generation, reconstruct `x̂₁` at each step, and track how far it moves,
`‖Δx̂₁‖`, normalized by per-step dynamism. A stable, well-conditioned flow **settles** (movement →0
as `t→1`); an under-conditioned one keeps oscillating late in sampling. Computed for both stages.
(The SS probe uses a single unweighted pass so the trajectory is a clean, unambiguous read of the
core denoiser.)


In [ ]:
# 8.0 — u_f: log the ODE solver's Euler steps, track clean-prediction movement
import torch, numpy as np, matplotlib.pyplot as plt

def _leaf(x):
    """Extract the latent tensor from the sampler state (tensor / dict / list)."""
    if torch.is_tensor(x):
        return x
    if isinstance(x, dict):
        for k in ("shape", "x"):
            if k in x and torch.is_tensor(x[k]):
                return x[k]
        return next(v for v in x.values() if torch.is_tensor(v))
    if isinstance(x, (list, tuple)):
        return x[0]
    return x

def record_trajectory(generator, run_fn):
    """Patch the generator's ODE solver to log (t, dt, x_t, x_{t+1}) for one generation."""
    solver = generator._solver
    orig_step = solver.step
    rec = []
    def patched(dynamics_fn, x_t, t, dt, *a, **k):
        x_tp1 = orig_step(dynamics_fn, x_t, t, dt, *a, **k)
        tt = float(t.item()) if torch.is_tensor(t) else float(t)
        dd = float(dt.item()) if torch.is_tensor(dt) else float(dt)
        rec.append((tt, dd, _leaf(x_t).detach().float().cpu(), _leaf(x_tp1).detach().float().cpu()))
        return x_tp1
    solver.step = patched
    try:
        run_fn()
    finally:
        solver.step = orig_step        # always restore, even on error
    return rec

def flow_instability(rec):
    """Reconstruct x̂₁ = x_t + (1-t)·v, v=(x_{t+1}-x_t)/dt; return (t, u_f_traj, dynamism)."""
    ts, x1hat, dyn = [], [], []
    for (t, dt, xt, xtp1) in rec:
        v = (xtp1 - xt) / max(dt, 1e-8)
        x1hat.append(xt + (1.0 - t) * v)
        dyn.append((xtp1 - xt).flatten().norm().item())
        ts.append(t)
    moves = [(x1hat[i + 1] - x1hat[i]).flatten().norm().item() for i in range(len(x1hat) - 1)]
    m = float(np.mean(dyn)) + 1e-8
    return np.array(ts[:-1]), np.array(moves) / m, np.array(dyn)

# ---- Stage 1 (SS): single unweighted pass = clean trajectory ----
_seed_all(SEED)
ss_rec = record_trajectory(ss_generator, lambda: pipeline.sample_sparse_structure_multi_view(
    ss_dicts_all, inference_steps=UF_STEPS, mode=MODE, ss_weighting=False))
ss_t, ss_uf, ss_dyn = flow_instability(ss_rec)
u_f_ss = float(ss_uf[len(ss_uf) // 2:].mean())     # late-sampling mean (instability = still moving late)
print(f"u_f (SS,   late-sampling mean) = {u_f_ss:.4f}  over {len(ss_rec)} Euler steps")

# ---- Stage 2 (SLAT) ----
u_f_slat = None
slat_t = slat_uf = None
if RUN_SLAT_METRICS:
    _seed_all(SEED)
    slat_rec = record_trajectory(slat_generator, lambda: pipeline.sample_slat_multi_view(
        slat_dicts_all, baseline_coords, inference_steps=STAGE2_STEPS, mode=MODE))
    slat_t, slat_uf, _slat_dyn = flow_instability(slat_rec)
    u_f_slat = float(slat_uf[len(slat_uf) // 2:].mean())
    print(f"u_f (SLAT, late-sampling mean) = {u_f_slat:.4f}  over {len(slat_rec)} Euler steps")

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(ss_t, ss_uf, "-o", ms=3, label="SS (Stage 1)")
if u_f_slat is not None:
    axs[0].plot(slat_t, slat_uf, "-s", ms=3, label="SLAT (Stage 2)")
axs[0].set_xlabel("flow time t  (0 = noise → 1 = data)")
axs[0].set_ylabel("u_f  =  ‖Δ clean-pred‖ / dynamism")
axs[0].set_title("Flow instability across denoising (should settle toward t=1)")
axs[0].legend(); axs[0].grid(alpha=.3)
axs[1].plot(ss_t, ss_dyn[:-1], "-o", ms=3, color="teal", label="SS raw step ‖Δx‖")
axs[1].set_xlabel("flow time t"); axs[1].set_title("Per-step dynamism (raw step size)")
axs[1].legend(); axs[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(HALLUC_OUT / "u_f_flow_instability.png", dpi=130); plt.show()


### 8.5 — Qualitative: watch the clean prediction settle across flow time

We decode the model's **clean-data prediction** `x̂₁` at t≈0.3, 0.6, 0.9 and paint each surface by
**how much it changed since the previous timestamp**. A stable, well-conditioned flow settles — the
coloring fades toward t≈0.9. If a region keeps lighting up late in sampling, that geometry is still
being "reinvented" each step (`u_f` high = unstable / hallucination-prone). Reuses the trajectory
already logged in §8 — no extra model runs.

In [ ]:
# 8.5 — Clean-prediction geometry at t≈0.3/0.6/0.9, colored by change since previous t
import numpy as np, matplotlib.pyplot as plt, torch
from skimage import measure

if "ss_rec" not in globals() or len(ss_rec) < 4:
    print("No SS flow trajectory (run §8 first) -> skipping.")
else:
    ts = np.array([r[0] for r in ss_rec])

    def _occ_at(t_target):
        i = int(np.argmin(np.abs(ts - t_target)))
        t, dt, xt, xtp1 = ss_rec[i]
        v = (xtp1 - xt) / max(dt, 1e-8)
        x1 = (xt + (1.0 - t) * v).to(_DEV)                       # clean-data prediction latent
        return float(ts[i]), ss_latent_to_occupancy(x1, as_prob=True)[0, 0].cpu().numpy()

    anchors = [0.15, 0.3, 0.6, 0.9]                              # index 0 is the "previous" for t≈0.3
    got = [_occ_at(a) for a in anchors]

    fig = plt.figure(figsize=(15, 5)); last = None
    for col, idx in enumerate([1, 2, 3]):
        t_now, occ_now   = got[idx]
        t_prev, occ_prev = got[idx - 1]
        change = np.abs(occ_now - occ_prev)                      # per-voxel change since previous anchor
        _lvl = 0.5 if occ_now.max() > 0.5 else float(occ_now.max()) * 0.5
        ax = fig.add_subplot(1, 3, col + 1, projection="3d")
        try:
            vv, ff, nn, _ = measure.marching_cubes(occ_now, level=_lvl)
            vi = np.clip(np.round(vv).astype(int), 0, 63)
            face_ch = change[vi[:, 0], vi[:, 1], vi[:, 2]][ff].mean(1)
            vv = vv / 64.0 - 0.5
            last = ax.plot_trisurf(vv[:, 0], vv[:, 1], vv[:, 2], triangles=ff, cmap="hot",
                                   linewidth=0, antialiased=False, shade=False)
            last.set_array(face_ch); last.set_clim(0, 0.5)
        except Exception as e:
            ax.text2D(0.5, 0.5, f"{e}", ha="center", transform=ax.transAxes)
        ax.view_init(elev=25, azim=-60); ax.set_box_aspect((1, 1, 1))
        ax.set_title(f"t≈{t_now:.2f}  (Δ since t≈{t_prev:.2f})", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    if last is not None:
        fig.colorbar(last, ax=fig.axes, shrink=0.5, label="|Δ occupancy| vs previous t")
    plt.suptitle("Metric 3 (qualitative) — clean prediction settling across flow time "
                 "(bright = still changing = unstable)")
    plt.savefig(HALLUC_OUT / "u_f_settling.png", dpi=130, bbox_inches="tight"); plt.show()

### 8.8 — Flow round-trip: the "noiser" (cycle-consistency + typicality)

The denoiser is a velocity field integrated **forward** (noise→data). The **noiser is the same field
integrated backward** (data→noise) — the flow ODE is a deterministic bijection, so we can renoise a
generated latent back to noise (no separate network). Two flow-native hallucination probes:

- **Cycle-consistency** — exactly the "noise → building → noise" idea: denoise `x0 → ẑ`, renoise
  `ẑ → x0'`, then re-denoise. `‖x0 − x0'‖` (noise space) and `‖ẑ − denoise(renoise(ẑ))‖` (latent space)
  measure how cleanly invertible the flow ODE is here. Small = well-conditioned; large = stiff /
  hallucination-prone. We paint the per-voxel latent residual on the surface.
- **Renoise typicality** — the backward map sends on-distribution latents to *typical* Gaussian noise,
  so we check whether the renoised `x0'` looks like an `N(0,I)` draw (`‖x0'‖²/D ≈ 1`, mean≈0, std≈1,
  Gaussian QQ). Off-distribution (hallucinated) latents renoise to atypical noise. Set
  `RUN_EXACT_LOGLIK=True` in the config for the exact (expensive) Hutchinson CNF log-likelihood.

Both use the **plain multidiffusion** flow — one shared velocity field for forward and backward — so
any residual is genuine ODE behavior, not a weighting artifact.

In [ ]:
# 8.8a — Flow cycle-consistency: denoise (noise→ẑ) → renoise (ẑ→noise) → re-denoise
import torch, numpy as np, matplotlib.pyplot as plt, matplotlib.cm as cm, trimesh
from skimage import measure
from sam3d_objects.pipeline.multi_view_utils import inject_generator_multi_view

# match the pipeline's SS generator setup; plain multidiffusion => ONE shared velocity field
ss_generator.no_shortcut = True
ss_generator.reverse_fn.strength = pipeline.ss_cfg_strength
ss_generator.reverse_fn.strength_pm = pipeline.ss_cfg_strength_pm
ss_generator.inference_steps = STAGE1_STEPS

cond_args, cond_kwargs = pipeline.get_multi_view_condition_input(
    pipeline.condition_embedders["ss_condition_embedder"], ss_dicts_all, pipeline.ss_condition_input_mapping)

B = 1
# This SS backbone (MoT sparse-structure flow) consumes a *dict* latent, so build the latent shape
# exactly like the pipeline and let the model make structured noise. We drive the EXACT generation
# path (same ss_generator(...) call the pipeline uses) but temporarily override (a) the initial state
# and (b) the time schedule. no_shortcut=True -> d=0 -> the plain velocity-field ODE, shared by the
# forward pass and its time-reversal (the "noiser").
if pipeline.is_mm_dit():
    latent_shape = {k: (B,) + (v.pos_emb.shape[0], v.input_layer.in_features)
                    for k, v in ss_generator.reverse_fn.backbone.latent_mapping.items()}
else:
    latent_shape = (B, 4096, 8)

t_fwd, d_val = ss_generator._prepare_t_and_d(STAGE1_STEPS)   # numpy schedule (0->1) + scalar step-size d
t_bwd = np.ascontiguousarray(t_fwd[::-1])                    # reversed schedule (data -> noise)
_orig_noise, _orig_prep = ss_generator._generate_noise, ss_generator._prepare_t_and_d

def _to_dev(x, device):
    return {k: v.to(device) for k, v in x.items()} if isinstance(x, dict) else x.to(device)

def _shape(x):
    return x["shape"] if isinstance(x, dict) else x

def run_flow(x_init, times):
    """Run the real SS generation from x_init over `times` (fused multi-view, d=0)."""
    ss_generator._generate_noise = lambda x_shape, x_device: _to_dev(x_init, x_device)
    ss_generator._prepare_t_and_d = lambda steps=None: (times, d_val)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=pipeline.shape_model_dtype):
            with inject_generator_multi_view(ss_generator, num_views=len(ss_dicts_all),
                                             num_steps=STAGE1_STEPS, mode="multidiffusion"):
                return ss_generator(latent_shape, _DEV, *cond_args, **cond_kwargs)
    finally:
        ss_generator._generate_noise, ss_generator._prepare_t_and_d = _orig_noise, _orig_prep

def _rel(a, b):
    a, b = _shape(a).float(), _shape(b).float()
    return float((a - b).flatten().norm() / (a.flatten().norm() + 1e-8))

torch.manual_seed(SEED)
x0 = ss_generator._generate_noise(latent_shape, _DEV)   # structured initial noise (seeded)
z_hat  = run_flow(x0, t_fwd)                             # denoise:    noise  -> latent
x0_re  = run_flow(z_hat, t_bwd)                          # renoise:    latent -> noise
z_hat2 = run_flow(x0_re, t_fwd)                          # re-denoise: noise  -> latent

r_noise  = _rel(x0, x0_re)                         # noise-space cycle error (your noise->bldg->noise)
r_latent = _rel(z_hat, z_hat2)                     # latent-space cycle error
print(f"Flow cycle-consistency (plain multidiffusion, {STAGE1_STEPS} steps):")
print(f"  ||x0 - renoise(denoise(x0))|| / ||x0||  = {r_noise:.4f}   (noise-space)")
print(f"  ||z - denoise(renoise(z))||   / ||z||   = {r_latent:.4f}   (latent-space)")
print("  small = flow ODE cleanly invertible here; large = stiff / hallucination-prone")

# empirical SS latent scale (report this instead of assuming std=1; sets the 'as-random' residual line)
_zc = _shape(z_hat).float()
_zstd = _zc.std().item()
print(f"SS latent empirical: std={_zstd:.4f} mean={_zc.mean().item():.4f} "
      f"min={_zc.min().item():.4f} max={_zc.max().item():.4f}")
print(f"  'as-random' cycle-residual level = sqrt(2)*std = {(2 ** 0.5) * _zstd:.3f} "
      f"(painted voxels above this ~= no cycle recovery)")

# per-voxel latent cycle residual -> paint on the decoded surface
resid16 = ((_shape(z_hat) - _shape(z_hat2)).float().permute(0, 2, 1).contiguous().view(B, 8, 16, 16, 16)
           .pow(2).mean(1).sqrt()[0].cpu().numpy())
ur64 = np.repeat(np.repeat(np.repeat(resid16, 4, 0), 4, 1), 4, 2)
occ = ss_latent_to_occupancy(_shape(z_hat), as_prob=True)[0, 0].cpu().numpy()
_lvl = 0.5 if occ.max() > 0.5 else float(occ.max()) * 0.5
try:
    verts, faces, normals, _ = measure.marching_cubes(occ, level=_lvl)
    vi = np.clip(np.round(verts).astype(int), 0, 63)
    vc = ur64[vi[:, 0], vi[:, 1], vi[:, 2]]
    vmax = float(np.percentile(vc, 99)) or 1.0
    vmesh = verts / 64.0 - 0.5
    trimesh.Trimesh(vertices=vmesh, faces=faces, vertex_normals=normals,
                    vertex_colors=(cm.viridis(np.clip(vc / vmax, 0, 1)) * 255).astype(np.uint8),
                    process=False).export(HALLUC_OUT / "flow_cycle_painted_mesh.glb")
    face_v = vc[faces].mean(1)
    fig = plt.figure(figsize=(15, 5)); last = None
    for i, (az, el) in enumerate([(-60, 25), (30, 25), (90, 80)]):
        ax = fig.add_subplot(1, 3, i + 1, projection="3d")
        last = ax.plot_trisurf(vmesh[:, 0], vmesh[:, 1], vmesh[:, 2], triangles=faces,
                               cmap="viridis", linewidth=0, antialiased=False, shade=False)
        last.set_array(face_v); last.set_clim(0, vmax)
        ax.view_init(elev=el, azim=az); ax.set_box_aspect((1, 1, 1))
        ax.set_title(f"flow cycle residual ({az},{el})", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    fig.colorbar(last, ax=fig.axes, shrink=0.5, label="latent cycle residual")
    plt.suptitle("Flow cycle-consistency painted on surface (bright = flow non-invertible / stiff)")
    plt.savefig(HALLUC_OUT / "flow_cycle_painted.png", dpi=130, bbox_inches="tight"); plt.show()
except Exception as e:
    print("cycle painting skipped:", repr(e))

In [ ]:
# 8.8b — Renoise typicality: is the renoised latent x0' a typical N(0, I) draw?
import numpy as np, matplotlib.pyplot as plt, torch

x = (x0_re["shape"] if isinstance(x0_re, dict) else x0_re).flatten().float().cpu().numpy()
D = x.size
norm2_over_D = float((x ** 2).mean())              # E[||x||^2]/D = 1 for standard normal
skew = float(((x - x.mean()) ** 3).mean() / (x.std() ** 3 + 1e-8))
kurt = float(((x - x.mean()) ** 4).mean() / (x.std() ** 4 + 1e-8) - 3.0)
print("Renoise typicality of ẑ (backward ODE  ẑ -> noise):")
print(f"  ||x0'||^2 / D = {norm2_over_D:.4f}   (≈1 typical;  »1 or «1 = off-distribution)")
print(f"  mean={x.mean():.4f} (≈0)  std={x.std():.4f} (≈1)  skew={skew:.3f} (≈0)  excess-kurt={kurt:.3f} (≈0)")

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].hist(x, bins=60, density=True, alpha=.8, color="slateblue", label="renoised x0'")
_g = np.linspace(-4, 4, 200)
axs[0].plot(_g, np.exp(-_g ** 2 / 2) / np.sqrt(2 * np.pi), "k--", label="N(0,1)")
axs[0].set_title("renoised-latent histogram vs standard normal"); axs[0].legend()
qs = np.linspace(0.01, 0.99, 99); samp = np.quantile(x, qs)
try:
    from scipy.stats import norm as _norm; theo = _norm.ppf(qs)
except Exception:
    theo = np.quantile(np.random.randn(400000), qs)
axs[1].plot(theo, samp, ".", ms=4); axs[1].plot([-3, 3], [-3, 3], "k--")
axs[1].set_xlabel("theoretical N(0,1) quantiles"); axs[1].set_ylabel("renoised quantiles")
axs[1].set_title("QQ plot (on the line = Gaussian = typical)")
plt.suptitle("Flow renoise typicality (proxy for log-likelihood under the flow prior)")
plt.tight_layout(); plt.savefig(HALLUC_OUT / "flow_renoise_typicality.png", dpi=130); plt.show()

# ---- optional: exact continuous-flow log-likelihood (expensive; per-view averaged) ----
RUN_EXACT_LOGLIK = globals().get("RUN_EXACT_LOGLIK", False)
exact_loglik = None
if RUN_EXACT_LOGLIK:
    try:
        lls = []
        for i in range(len(ss_dicts_all)):
            vc_args, vc_kwargs = pipeline.get_multi_view_condition_input(
                pipeline.condition_embedders["ss_condition_embedder"], [ss_dicts_all[i]],
                pipeline.ss_condition_input_mapping)
            with torch.autocast(device_type="cuda", dtype=pipeline.shape_model_dtype):
                ll = ss_generator.log_likelihood(z_hat, steps=max(8, STAGE1_STEPS // 2),
                                                 z_samples=1, *vc_args, **vc_kwargs)
            lls.append(float(ll.mean()))
        exact_loglik = float(np.mean(lls))
        print(f"Exact CNF log-likelihood (per-view mean) = {exact_loglik:.2f}  (higher = more typical)")
    except Exception as e:
        print("Exact log-likelihood skipped (expensive / needs grad):", repr(e))
else:
    print("RUN_EXACT_LOGLIK=False -> cheap renoise-typicality proxy only "
          "(set RUN_EXACT_LOGLIK=True in the config cell for the exact Hutchinson CNF log-likelihood).")

## 9 — Hallucination report

Collect the scalar predictors for this scene (Stage 1 and Stage 2), save a JSON + summary figure.
These are **runtime, label-free** uncertainty scores: they don't need ground truth. In the paper each
correlates with realized rollout error at Spearman ρ≈0.8; here they flag which stage / which geometry
the base MV-SAM3D model is least certain about.


In [ ]:
# 9.0 — Assemble + save the hallucination report
import json, numpy as np, matplotlib.pyplot as plt

def _g(name):
    return globals().get(name, None)

report = {
    "scene": DATA_ROOT.name,
    "views": int(N_VIEWS),
    "aerial_views": int(len(aerial_images)),
    "ground_views": int(len(ground_images)),
    "seeds": list(SEED_LIST),
    "stage1_steps": int(STAGE1_STEPS),
    "stage2_steps": int(STAGE2_STEPS),
    "metrics": {
        "u_r_ss":            _g("u_r"),
        "u_r_ss_random_ref": _g("u_r_rand"),
        # normalized 0->1 on-manifold score: 0 = fully on-manifold, ~1 = as off-manifold as pure noise
        "u_r_ss_onmanifold_ratio": (float(_g("u_r") / _g("u_r_rand"))
                                    if (_g("u_r") is not None and _g("u_r_rand")) else None),
        "u_s_ss":            _g("u_s_ss"),   # per-voxel occupancy std, bounded in [0, 0.5]
        "u_s_slat":          _g("u_s_slat"),
        "u_f_ss":            _g("u_f_ss"),   # dynamism-normalized (dimensionless, per paper u/m)
        "u_f_slat":          _g("u_f_slat"),
        # flow-native probes (§8.8): ODE invertibility + renoise typicality
        "flow_cycle_noise":  _g("r_noise"),          # ||x0 - renoise(denoise(x0))|| / ||x0||
        "flow_cycle_latent": _g("r_latent"),         # ||z  - denoise(renoise(z))||  / ||z||
        "flow_renoise_norm2_over_D": _g("norm2_over_D"),  # ≈1 = renoised latent is typical N(0,I)
        "flow_exact_loglik": _g("exact_loglik"),     # None unless RUN_EXACT_LOGLIK=True
    },
}
print(json.dumps(report, indent=2))
(HALLUC_OUT / "hallucination_report.json").write_text(json.dumps(report, indent=2))

labels, vals, colors = [], [], []
_palette = {"u_r_ss": "crimson", "u_s_ss": "darkorange", "u_s_slat": "purple",
            "u_f_ss": "teal", "u_f_slat": "seagreen"}
for k in ("u_r_ss", "u_s_ss", "u_s_slat", "u_f_ss", "u_f_slat"):
    v = report["metrics"][k]
    if v is not None:
        labels.append(k); vals.append(v); colors.append(_palette[k])
if vals:
    plt.figure(figsize=(9, 4))
    plt.bar(labels, vals, color=colors, alpha=.85)
    for i, v in enumerate(vals):
        plt.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    plt.title(f"MV-SAM3D hallucination predictors — {report['scene']} ({N_VIEWS} views)")
    plt.ylabel("metric value"); plt.grid(axis="y", alpha=.3)
    plt.tight_layout(); plt.savefig(HALLUC_OUT / "hallucination_metrics.png", dpi=140); plt.show()
print("\nSaved report + figures to", HALLUC_OUT)


In [ ]:
# 9.5 — Copy the hallucination results to Google Drive (persists after the session ends)
import shutil
from pathlib import Path

if Path("/content/drive/MyDrive").exists():
    DRIVE_RESULTS = Path("/content/drive/MyDrive/mvsam3d_hallucination") / DATA_ROOT.name
    if DRIVE_RESULTS.exists():
        shutil.rmtree(DRIVE_RESULTS)
    shutil.copytree(HALLUC_OUT, DRIVE_RESULTS)
    print("Copied results to Drive:", DRIVE_RESULTS)
    for f in sorted(DRIVE_RESULTS.iterdir()):
        print("  ", f.name, f"({f.stat().st_size/1e6:.2f} MB)")
else:
    print("Drive not mounted at /content/drive/MyDrive — skipped (run the drive.mount cell first).")
    print("Local results are in:", HALLUC_OUT)


## 10 — Notes, interpretation & limitations

- **Two checkpoints, as requested.** Every metric is reported *after Stage 1* (SS geometry) and,
  where well-defined, *after both stages* (SLAT). `u_s` and `u_f` cover both stages; `u_r` covers
  Stage 1 only.
- **Why `u_r` is Stage-1-only.** The round trip needs a clean latent→data→latent inverse. The SS
  VAE gives exactly that (`E_S ∘ D_S`), and the SS generator emits latents in the encoder's space
  (no de-normalization). The SLAT decoder instead emits meshes/gaussians (not re-ingested by the
  SLAT encoder), and SLAT latents are rescaled by `slat_std/slat_mean`, so a faithful SLAT round
  trip is ill-posed without rendering + re-encoding.
- **Determinism.** Given a fixed seed the sampler is deterministic, so `u_r`/`u_f` are reproducible
  and `u_s` isolates exactly the effect of the initial random latent.
- **Dynamism normalization.** Following the paper we report `u_f` (and, for the active region,
  `u_s`) relative to per-step / mean scene activity, so the scores are scale-free.
- **These are relative, per-scene scores** — compare across scenes/views/seeds, or against the
  random-latent reference for `u_r`. They are not absolute physical errors.
- **Cost.** `u_s` re-runs the sampler `len(SEED_LIST)` times per stage. Bump `SEED_LIST` for tighter
  variance estimates; set `RUN_SLAT_METRICS=False` for a fast geometry-only pass.
- **Flow-native probes (§8.8) vs. `u_r`.** `u_r` is a *VAE* round trip (latent→occupancy→latent).
  The §8.8 probes instead exercise the *flow* itself: the "noiser" is the same velocity field
  integrated backward (data→noise), so `flow_cycle_*` measures ODE invertibility / stiffness and
  `flow_renoise_norm2_over_D≈1` checks that a generated latent renoises to *typical* Gaussian noise
  (a cheap stand-in for the flow log-likelihood). `u_f` and `u_s` are already flow-process metrics
  (per-step prediction drift; sensitivity to the initial noise); only `u_r` lives in VAE space.
- **`RUN_EXACT_LOGLIK`.** Off by default. When `True`, §8.8b runs the model's Hutchinson-trace
  continuous-flow `log_likelihood` (autograd through the DiT, per-view averaged) — the exact
  typicality score, but minutes-scale and memory-heavy.
